<p class="lead">Este notebook desarrolla la cuarta unidad del curso en el lenguaje R. Las diapositivas de clase son un mapa; este texto es el territorio. La unidad comienza con una <em>estructura de datos</em> —el árbol binario, tema de un primer curso de algoritmos— y termina con uno de los métodos de aprendizaje más exitosos de los últimos veinticinco años, el bosque aleatorio. Las dos mitades van juntas: un árbol de decisión <em>es</em> un árbol binario cuyos nodos internos hacen preguntas sobre las características y cuyas hojas contienen predicciones, y un bosque aleatorio es una colección de tales árboles, crecidos sobre datos aleatorizados y promediados. Lo que las Secciones 2–5 establecen sobre profundidad, altura, recorrido y balance se usa literalmente en las Secciones 6–8, con los datos de cáncer de mama que reaparecen a lo largo de todo el curso. Cada figura y cada número del texto se producen con el código R que puedes ver y modificar.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Definir recursivamente un árbol binario; usar correctamente raíz, hoja, profundidad, altura y grado; contar los nodos de un árbol perfecto de altura \(h\).</li>
<li>Implementar un árbol binario como nodos enlazados (environments de R), como vector (regla \(2i+1\) / \(2i+2\)) y como lista anidada, e implementar los cuatro recorridos.</li>
<li>Enunciar el invariante del árbol binario de búsqueda, insertar y buscar en \(O(h)\), y explicar por qué el orden de inserción determina la altura.</li>
<li>Explicar por qué importa el balance, qué hace una rotación y qué garantizan los árboles AVL, rojo–negro y B.</li>
<li>Describir un árbol de decisión como un árbol binario sobre el espacio de características; derivar las impurezas de Gini, de entropía y de error de clasificación; encontrar la mejor división de una característica a mano y con <code>rpart</code>.</li>
<li>Diagnosticar el sobreajuste con curvas de profundidad y de poda; leer las reglas de un árbol y sus importancias de características con las salvedades apropiadas.</li>
<li>Explicar el bagging, los subconjuntos aleatorios de características y la estimación out-of-bag; comparar un bosque aleatorio con un solo árbol y con los métodos de la Unidad 3 bajo la metodología del Suplemento S1.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Léelo de corrido una vez, ejecutando cada celda de código, y luego vuelve a él por secciones. Cada celda de código es breve y está comentada, y el párrafo que la sigue interpreta su salida. Los ejercicios (cajas verdes) te piden modificar la celda anterior y volver a ejecutarla. Las cajas azules contienen ideas clave, las naranjas errores frecuentes y las moradas conectan con otras unidades o con el curso de bioinformática (BGM324). Solo se usan R base y los paquetes <code>rpart</code>, <code>rpart.plot</code>, <code>randomForest</code>, <code>mlbench</code>, <code>e1071</code>, <code>class</code> y <code>pROC</code>; en un entorno de ejecución de R recién iniciado en Google Colab, ejecuta una vez <code>install.packages(c("rpart.plot","randomForest","mlbench","e1071","pROC"))</code>. No se descargan datos. Si ya cursaste estructuras de datos, lee rápido las Secciones 2–4, pero no te saltes el ejemplo resuelto de la Sección 5.</p></div>



In [ ]:
# Configuración: paquetes usados en todo el notebook. Las semillas se fijan para que cada número del texto sea reproducible.
suppressPackageStartupMessages({
  library(rpart); library(rpart.plot); library(randomForest)
  library(mlbench); library(e1071); library(class); library(pROC)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"
cat("R", as.character(getRversion()), "| rpart", as.character(packageVersion("rpart")),
    "| randomForest", as.character(packageVersion("randomForest")),
    "| mlbench", as.character(packageVersion("mlbench")), "\n")



La salida registra las versiones con las que se construyeron estas notas. `set.seed(0)` fija el generador global; las celdas estocásticas de más abajo, además, vuelven a fijar su propia semilla, de modo que cada número es reproducible por sí solo.

## 1. Por qué un curso de inteligencia artificial enseña una estructura de datos

Un árbol es la representación natural de una <strong>decisión jerárquica</strong>. Un clínico que hace el triaje de un dolor torácico plantea una primera pregunta (¿es anormal el electrocardiograma?), la respuesta determina la siguiente, y así sucesivamente hasta llegar a una decisión. Puesto por escrito, el protocolo es un árbol: cada pregunta un nodo, cada respuesta una rama, cada decisión una hoja. El primer protocolo de ese tipo derivado por computador, construido en 1982 a partir de 482 pacientes de urgencias con el método de partición recursiva que esta unidad llama CART, redujo los ingresos innecesarios a unidades coronarias sin pasar por alto más infartos que los médicos.<sup class="c"><a href="#r27">[27]</a></sup>

La misma estructura aparece en toda la ciencia de la computación por otra razón: un árbol que reduce a la mitad las posibilidades restantes en cada nodo alcanza cualquiera de \(n\) elementos en unos \(\log_2 n\) pasos: veinte pasos para un millón de registros. El tratamiento de Knuth de los árboles y la búsqueda sigue siendo la referencia para el vocabulario y el análisis.<sup class="c"><a href="#r1">[1]</a>,<a href="#r2">[2]</a></sup> La unidad avanza en tres pasos (Figura 1): el <strong>árbol binario</strong> como estructura de datos (Secciones 2–5); el <strong>árbol de decisión</strong> (Sección 6), un árbol binario cuyos nodos —umbrales sobre características— se aprenden de los datos y cuya profundidad controla cuánto memoriza; y el <strong>bosque aleatorio</strong> (Sección 7), cientos de árboles de decisión crecidos sobre remuestreos aleatorios de los pacientes y subconjuntos aleatorios de las características, que votan en conjunto.

<div class="box key"><h4>El hilo de la unidad</h4><p>Cada propiedad de un árbol binario vista en las Secciones 2–5 tiene una consecuencia para el aprendizaje. La <em>altura</em> es el número de preguntas que se hacen antes de una predicción y, en un árbol aprendido, el principal control del sobreajuste. El <em>balance</em> decide si una predicción cuesta tiempo logarítmico o lineal. El <em>recorrido</em> es la forma en que un árbol se imprime como reglas, y la forma en que <code>rpart</code> lo almacena: como un data frame indexado por nodo, la representación en vector de la Sección 2.3.</p></div>



In [ ]:
# Figura 1: las tres etapas de la unidad (árbol binario -> árbol de decisión -> bosque), dibujadas con gráficos base.
draw_full <- function(labels, fills, main, fs = 0.7) {
  P <- list(c(0,0), c(-1,-1), c(1,-1), c(-1.5,-2), c(-0.5,-2), c(0.5,-2), c(1.5,-2))
  plot.new(); plot.window(xlim = c(-2.3, 2.3), ylim = c(-2.5, 0.5))
  for (i in 2:7) { p <- (i - 2) %/% 2 + 1; segments(P[[p]][1], P[[p]][2], P[[i]][1], P[[i]][2], col = GREY) }
  for (i in 1:7) { points(P[[i]][1], P[[i]][2], pch = 21, bg = fills[i], cex = 4.6); text(P[[i]][1], P[[i]][2], labels[i], cex = fs) }
  title(main = main, cex.main = 0.9)
}
par(mfrow = c(1, 3), mar = c(1, 1, 2, 1))
draw_full(LETTERS[1:7], rep("white", 7), "1. A binary tree (Sec. 2-5)")
draw_full(c("shape>3?","nuclei>5?","size>4?","benign","malig.","benign","malig."),
          c(rep("#dfe7f1",3), "#e3f1e3","#fbe3d3","#e3f1e3","#fbe3d3"), "2. A decision tree (Sec. 6)", fs = 0.6)
draw_full(c("?","?","?","B","M","M","B"), c(rep("#dfe7f1",3), rep("#e3f1e3",4)), "3. One tree of a forest (Sec. 7)")



<p class="caption"><b>Figura 1.</b> Los tres objetos de la unidad: un árbol binario abstracto (el ejemplo recurrente de la Sección 2); un árbol de decisión sobre características de cáncer de mama —la misma forma, con una pregunta de sí/no en cada nodo interno y un diagnóstico en cada hoja—; y un árbol de un bosque cuyos votos se combinan.</p>

## 2. Árboles binarios

### 2.1 Definiciones

Un <strong>árbol</strong> es un conjunto de nodos conectados por aristas con exactamente un camino entre dos nodos cualesquiera; un <strong>árbol con raíz</strong> distingue un nodo, la <strong>raíz</strong>, desde el cual se llega a todos los demás nodos "bajando". El vocabulario, debido en gran parte a Knuth, mezcla genealogía y botánica:<sup class="c"><a href="#r1">[1]</a></sup>

| Término | Definición | En la Figura 1 (izquierda) |
|---|---|---|
| **Raíz** | El único nodo sin padre | A |
| **Padre / hijo** | El nodo inmediatamente arriba / abajo a lo largo de una arista | B es el padre de D y E |
| **Hoja** | Un nodo sin hijos | D, E, F, G |
| **Nodo interno** | Un nodo con al menos un hijo | A, B, C |
| **Grado** de un nodo | Su número de hijos | 2 para todo nodo interno |
| **Profundidad** de un nodo | Aristas desde la raíz; la raíz tiene profundidad 0 | depth(D) = 2 |
| **Altura** de un nodo | Aristas del camino más largo hacia abajo hasta una hoja; una hoja tiene altura 0 | height(B) = 1 |
| **Altura** del árbol | Altura de la raíz = profundidad máxima | 2 |
| **Subárbol** con raíz en \(v\) | \(v\) y todos sus descendientes | {B, D, E} |

Un <strong>árbol binario</strong> es un árbol con raíz en el que cada nodo tiene a lo sumo dos hijos, distinguidos como <strong>izquierdo</strong> y <strong>derecho</strong>. Es <strong>lleno</strong> si cada nodo tiene 0 o 2 hijos, <strong>perfecto</strong> si es lleno y todas sus hojas tienen la misma profundidad (Figura 1, izquierda), y <strong>completo</strong> si todos los niveles salvo el último están llenos y el último se llena desde la izquierda: la forma de los heaps (montículos) y de la representación en vector que se presenta más abajo.

<div class="box warn"><h4>La altura se cuenta en aristas; el árbol vacío tiene altura −1</h4><p>Algunos textos cuentan nodos en lugar de aristas, de modo que un nodo solo tiene altura 1 en vez de 0. Este notebook y la mayoría de los textos de algoritmos cuentan aristas y asignan al árbol vacío altura −1, para que la fórmula recursiva de abajo se cumpla sin excepciones. Un error por uno convierte \(2^{h+1}-1\) en \(2^{h}-1\).</p></div>

### 2.2 La definición recursiva

<em>Un árbol binario es o bien vacío, o bien un nodo raíz junto con dos árboles binarios, sus subárboles izquierdo y derecho.</em> Todo algoritmo sobre árboles sigue esta forma —procesar la raíz, recurrir a la izquierda, recurrir a la derecha— y toda propiedad se demuestra por inducción sobre la altura. Por ejemplo

\[ h(T) = \begin{cases} -1 & T \text{ empty} \\ 1 + \max\big(h(T_\text{left}), h(T_\text{right})\big) & \text{otherwise,} \end{cases} \]

y \(n(T) = 0\) para el árbol vacío, \(1 + n(T_\text{left}) + n(T_\text{right})\) en otro caso.

### 2.3 Tres representaciones

La primera representación son los <strong>nodos enlazados</strong>. Los objetos ordinarios de R se copian al asignarse, así que, para obtener la semántica de referencia de una estructura enlazada, guardamos cada nodo en un <strong>environment</strong> (entorno): un contenedor mutable con un valor y dos referencias, <code>left</code> y <code>right</code>, que valen <code>NULL</code> cuando falta un hijo. La celda construye así el árbol de la Figura 1 y calcula su altura y tamaño con las definiciones recursivas.



In [ ]:
# Representación 1: nodos enlazados como environments (semántica de referencia, como un árbol basado en punteros).
node <- function(v, l = NULL, r = NULL) { e <- new.env(); e$val <- v; e$left <- l; e$right <- r; e }

tree_height <- function(t) {              # altura en aristas, iterativa (segura para los árboles profundos de las Secc. 3-4)
  if (is.null(t)) return(-1L)
  st <- list(list(n = t, d = 0L)); h <- -1L
  while (length(st)) {
    x <- st[[length(st)]]; st[[length(st)]] <- NULL; h <- max(h, x$d)
    if (!is.null(x$n$left))  st[[length(st) + 1]] <- list(n = x$n$left,  d = x$d + 1L)
    if (!is.null(x$n$right)) st[[length(st) + 1]] <- list(n = x$n$right, d = x$d + 1L)
  }
  h
}
tree_count  <- function(t) if (is.null(t)) 0L else 1L + tree_count(t$left) + tree_count(t$right)
tree_leaves <- function(t) if (is.null(t)) c() else if (is.null(t$left) && is.null(t$right)) t$val else c(tree_leaves(t$left), tree_leaves(t$right))

# El árbol de siete nodos de la Figura 1 (izquierda), construido de abajo hacia arriba: primero las hojas, luego los padres y luego la raíz.
T <- node("A", node("B", node("D"), node("E")), node("C", node("F"), node("G")))
cat("root:", T$val, "| left child of root:", T$left$val, "| right child of B:", T$left$right$val, "\n")
cat("height:", tree_height(T), "| nodes:", tree_count(T), "| leaves:", paste(tree_leaves(T), collapse = " "), "\n")
cat("height of subtree at B:", tree_height(T$left), "| of a leaf:", tree_height(T$left$left), "| of the empty tree:", tree_height(NULL), "\n")



Las funciones son la definición recursiva convertida en código, y las salidas confirman el conteo a mano: altura 2, siete nodos, cuatro hojas, altura(hoja) = 0, altura(vacío) = −1.

La segunda representación es un <strong>vector</strong>: se numeran los nodos nivel por nivel, con la raíz en 0; los hijos del nodo \(i\) están en \(2i+1\) y \(2i+2\), y su padre en \(\lfloor (i-1)/2 \rfloor\). No se almacenan referencias: la forma está en la aritmética. Es compacta para árboles completos (el heap binario es exactamente esto<sup class="c"><a href="#r2">[2]</a></sup>) y derrochadora en otro caso, porque los nodos faltantes igual ocupan posiciones. La tercera es una <strong>lista anidada</strong>, cómoda para serializar a JSON.



In [ ]:
# Representación 2: un vector por niveles (hijos de i en 2i+1, 2i+2, contando desde 0). Representación 3: una lista anidada.
to_vector <- function(t) {
  arr <- rep(NA_character_, 2 ^ (tree_height(t) + 1) - 1)
  fill <- function(nd, i) { if (is.null(nd)) return(); arr[i + 1] <<- nd$val; fill(nd$left, 2 * i + 1); fill(nd$right, 2 * i + 2) }
  fill(t, 0); arr
}
to_list <- function(t) if (is.null(t)) NULL else list(value = t$val, left = to_list(t$left), right = to_list(t$right))

arr <- to_vector(T)
cat("vector (0-based index shown):", paste0(0:(length(arr) - 1), ":", arr, collapse = "  "), "\n")
for (i in 0:2) cat(sprintf("  node %s at index %d: children at %d (%s) and %d (%s); parent of %d is %d (%s)\n",
    arr[i + 1], i, 2 * i + 1, arr[2 * i + 2], 2 * i + 2, arr[2 * i + 3], 2 * i + 1, (2 * i) %/% 2, arr[(2 * i) %/% 2 + 1]))
cat("nested list, left spine:", to_list(T)$value, "->", to_list(T)$left$value, "->", to_list(T)$left$left$value, "\n")



El vector se lee `A B C D E F G`, el listado por niveles, y la aritmética recupera todas las relaciones padre–hijo. La Sección 6.1 muestra que <code>rpart</code> almacena un árbol aprendido con esta misma idea: un data frame cuyos nombres de fila son números de nodo que obedecen la misma regla (hijos del nodo \(i\) en \(2i\) y \(2i+1\), contando desde uno), de modo que los árboles desbalanceados se almacenan sin desperdiciar posiciones.

### 2.4 Recorridos

<strong>Recorrer</strong> un árbol es visitar cada nodo una vez en un orden definido. La definición recursiva ofrece tres lugares para visitar la raíz —antes, entre o después de las llamadas recursivas— más un orden en anchura:<sup class="c"><a href="#r1">[1]</a></sup>

| Recorrido | Orden | Uso típico |
|---|---|---|
| **Preorden** | raíz, izquierdo, derecho | imprimir un árbol como reglas anidadas (Sección 6) |
| **Inorden** | izquierdo, raíz, derecho | en un árbol de búsqueda, entrega las claves ordenadas (Sección 3) |
| **Postorden** | izquierdo, derecho, raíz | eliminar un árbol (primero los hijos); evaluar árboles de expresiones |
| **Por niveles** | profundidad 0, luego profundidad 1, … | la representación en vector |

A mano sobre la Figura 1: el preorden visita A, luego el subárbol de B (B, D, E) y luego el de C: `A B D E C F G`; el inorden da `D B E A F C G`; el postorden `D E B F G C A`; y el recorrido por niveles `A B C D E F G`.



In [ ]:
# Los cuatro recorridos. Los tres en profundidad difieren solo en DÓNDE se visita la raíz.
preorder  <- function(t) if (is.null(t)) c() else c(t$val, preorder(t$left),  preorder(t$right))
inorder   <- function(t) if (is.null(t)) c() else c(inorder(t$left),  t$val, inorder(t$right))
postorder <- function(t) if (is.null(t)) c() else c(postorder(t$left), postorder(t$right), t$val)
levelorder <- function(t) {                      # en anchura: una cola de nodos descubiertos pero no visitados
  if (is.null(t)) return(c())
  out <- c(); q <- list(t)
  while (length(q)) { nd <- q[[1]]; q <- q[-1]; out <- c(out, nd$val)
    if (!is.null(nd$left))  q <- c(q, list(nd$left))
    if (!is.null(nd$right)) q <- c(q, list(nd$right)) }
  out
}
for (nm in c("pre-order", "in-order", "post-order", "level-order"))
  cat(sprintf("%-12s %s\n", nm, paste(switch(nm, "pre-order" = preorder, "in-order" = inorder,
      "post-order" = postorder, levelorder)(T), collapse = " ")))



Los cuatro coinciden con la traza a mano, y lo único que separa los tres recorridos en profundidad es la posición de `t$val`. El inorden es el importante: en un árbol de búsqueda entrega las claves ordenadas.

### 2.5 Conteo de nodos

Un árbol <strong>perfecto</strong> de altura \(h\) tiene \(2^d\) nodos a profundidad \(d\), por lo que

\[ n = \sum_{d=0}^{h} 2^d = 2^{h+1} - 1, \qquad h = \log_2(n+1) - 1, \]

con \(2^h\) hojas. A la inversa, cualquier árbol binario con \(n\) nodos tiene altura al menos \(\lceil \log_2(n+1) \rceil - 1\) y a lo sumo \(n-1\) (una cadena). Estas dos cotas, logarítmica y lineal, son toda la historia de las Secciones 3 y 4.



In [ ]:
# Árboles binarios perfectos de altura 0..6: contar nodos y hojas y comparar con 2^(h+1)-1 y 2^h.
perfect <- function(h, label = 1) if (h < 0) NULL else node(label, perfect(h - 1, 2 * label), perfect(h - 1, 2 * label + 1))
tab <- do.call(rbind, lapply(0:6, function(h) {
  t <- perfect(h)
  data.frame(height_h = h, nodes = tree_count(t), fml_2h1_minus1 = 2 ^ (h + 1) - 1,
             leaves = length(tree_leaves(t)), fml_2h = 2 ^ h, height_recomputed = tree_height(t))
}))
print(tab, row.names = FALSE)



Los conteos concuerdan con la fórmula: 1, 3, 7, 15, 31, 63, 127. Leído al revés, un árbol perfecto que contiene un millón de elementos tiene altura de solo 19, de modo que cualquier pregunta sobre su contenido se responde en unos veinte pasos, <em>siempre que el árbol se mantenga cerca de ser perfecto</em>.

<div class="exercise"><h4>Ejercicio 1</h4><p>Modifica <code>perfect()</code> para que cada nodo tenga solo un hijo <em>derecho</em>. ¿Cuántos nodos tiene una cadena de altura \(h\), y cuántas posiciones desperdicia <code>to_vector()</code> para una cadena de altura 4? Esta es la forma degenerada de la Sección 3.4.</p></div>

## 3. Árboles binarios de búsqueda

### 3.1 El invariante

Un <strong>árbol binario de búsqueda</strong> (BST, binary search tree) contiene <em>claves</em> de un conjunto ordenado y satisface, en cada nodo \(v\), el <strong>invariante BST</strong>:

\[ \text{every key in the left subtree of } v \;<\; \text{key}(v) \;<\; \text{every key in the right subtree of } v , \]

una condición sobre subárboles completos, no solo sobre los dos hijos. Aquí no se permiten claves duplicadas.

Para <strong>buscar</strong> \(k\): se parte en la raíz; se detiene si \(k\) es igual a la clave del nodo; se va a la izquierda si es menor y a la derecha si es mayor; si el hijo no existe, \(k\) no está en el árbol. Para <strong>insertar</strong> \(k\): se busca y se adjunta una hoja nueva donde la búsqueda se salió del árbol. Cada paso desciende un nivel, de modo que ambas operaciones cuestan a lo sumo la altura: \(O(h)\). Lo que importa es cuán grande es \(h\) en relación con \(n\).<sup class="c"><a href="#r2">[2]</a></sup> La implementación guarda el árbol en un environment contenedor y cuenta las comparaciones de claves para los experimentos de costo de las Secciones 3.4 y 4.5; la inserción y la búsqueda se escriben como bucles (no como recursión), de modo que son seguras en los árboles profundos que vendrán.



In [ ]:
# Un árbol binario de búsqueda con claves distintas; cuenta comparaciones; la inserción y la búsqueda son iterativas.
new_bst <- function() { b <- new.env(); b$root <- NULL; b$comparisons <- 0L; b }
bst_insert <- function(b, key) {
  if (is.null(b$root)) { b$root <- node(key); return(invisible()) }
  cur <- b$root
  repeat {
    b$comparisons <- b$comparisons + 1L
    if (key < cur$val)      { if (is.null(cur$left))  { cur$left  <- node(key); break }; cur <- cur$left }
    else if (key > cur$val) { if (is.null(cur$right)) { cur$right <- node(key); break }; cur <- cur$right }
    else break                                           # las claves iguales se ignoran
  }
  invisible()
}
bst_search <- function(b, key) {                         # devuelve el camino de claves visitadas
  path <- c(); cur <- b$root
  while (!is.null(cur)) { path <- c(path, cur$val); b$comparisons <- b$comparisons + 1L
    if (key == cur$val) return(list(path = path, found = TRUE))
    cur <- if (key < cur$val) cur$left else cur$right }
  list(path = path, found = FALSE)
}
bst <- new_bst()
for (k in c(41, 20, 65, 11, 29, 50, 91, 32, 72, 99)) bst_insert(bst, k)
cat("in-order traversal:", paste(inorder(bst$root), collapse = " "), "\n")
cat("height:", tree_height(bst$root), "| nodes:", tree_count(bst$root), "\n")
for (k in c(32, 72, 60)) { s <- bst_search(bst, k)
  cat(sprintf("search %d: path %s -> %s after %d comparisons\n", k, paste(s$path, collapse = " "),
      if (s$found) "found" else "absent", length(s$path))) }



El recorrido inorden devuelve las claves en orden ascendente aunque se insertaron sin ningún orden en particular: el invariante hecho visible, y la razón por la que funciona el <em>ordenamiento por árbol</em> (tree sort). Las diez claves produjeron altura 3, el mínimo \(\lceil \log_2 11 \rceil - 1\), y cada búsqueda se detiene tras a lo sumo \(h+1 = 4\) comparaciones, encuentre o no la clave.

### 3.2 Por qué el inorden entrega las claves ordenadas

Por inducción sobre la altura: el inorden visita el subárbol izquierdo, la raíz y luego el subárbol derecho; por el invariante, toda clave de la izquierda es menor que la raíz y toda clave de la derecha es mayor; por la hipótesis inductiva, cada subárbol se visita en orden; la concatenación queda ordenada. El mismo argumento da una verificación del invariante en \(O(n)\): recorrer en inorden y comprobar que la secuencia es creciente.

### 3.3 Eliminación, a grandes rasgos

Una <em>hoja</em> se elimina; un nodo con <em>un hijo</em> se reemplaza por ese hijo; un nodo con <em>dos hijos</em> recibe la clave de su <strong>sucesor inorden</strong> (la clave más pequeña de su subárbol derecho), y en su lugar se elimina el sucesor, que tiene a lo sumo un hijo. El invariante se preserva y el costo es \(O(h)\). Knuth señala que las eliminaciones repetidas con esta regla desbalancean lentamente el árbol, una de las motivaciones de la Sección 4.<sup class="c"><a href="#r2">[2]</a></sup>

### 3.4 Degeneración: la altura depende del orden de inserción

Nada en el algoritmo de inserción mantiene bajo el árbol. Si las claves llegan ordenadas, cada clave nueva va a la derecha: el árbol es una cadena de altura \(n-1\) y la búsqueda es un recorrido lineal disfrazado de árbol. Si las claves llegan en orden aleatorio, la raíz divide el resto aproximadamente por la mitad, cada subárbol es a su vez un BST aleatorio y la altura esperada es proporcional a \(\log n\).<sup class="c"><a href="#r2">[2]</a></sup> Una entrada ordenada no es un caso artificial: los identificadores de pacientes, las marcas de tiempo y los números de acceso se generan en orden creciente.



In [ ]:
# Altura de un BST tras insertar n claves en orden (degenerado) frente a orden aleatorio (20 permutaciones por n).
set.seed(0)
ns <- c(10, 20, 50, 100, 200, 500, 1000, 2000)
res <- do.call(rbind, lapply(ns, function(n) {
  st <- new_bst(); for (k in 0:(n - 1)) bst_insert(st, k)          # entrada ordenada -> cadena
  rh <- replicate(20, { t <- new_bst(); for (k in sample(0:(n - 1))) bst_insert(t, k); tree_height(t$root) })
  data.frame(n = n, sorted = tree_height(st$root), random_mean = mean(rh), random_max = max(rh), log2n = log2(n))
}))
plot(res$n, res$sorted, log = "xy", type = "b", pch = 15, col = ORANGE, xlab = "number of keys n",
     ylab = "height of the tree", main = "BST height versus number of keys and insertion order")
lines(res$n, res$random_mean, type = "b", pch = 19, col = BLUE)
lines(res$n, res$log2n, lty = 2, col = GREY)
legend("topleft", bty = "n", cex = 0.8, col = c(ORANGE, BLUE, GREY), pch = c(15, 19, NA), lty = c(1, 1, 2),
       legend = c("sorted insertion: h = n - 1", "random insertion: mean of 20 trees", "log2(n), perfect-tree bound"))
print(round(res, 1), row.names = FALSE)



<p class="caption"><b>Figura 2.</b> Altura de un árbol binario de búsqueda frente al número de claves, en ejes logarítmicos. La inserción ordenada da \(h = n-1\); la inserción aleatoria da alturas que crecen como una constante por \(\log_2 n\). En línea discontinua: la cota inferior del árbol perfecto.</p>

Los árboles aleatorios son dos a tres veces más altos que un árbol perfecto —el precio de no controlar la forma—, pero siguen siendo logarítmicos, mientras que la cadena es lineal: unas dos docenas de comparaciones por búsqueda para \(n = 2{,}000\), donde la cadena necesita hasta 2,000. Un BST es tan rápido como lo permite su forma, y la forma queda a merced de los datos a menos que el algoritmo la mantenga.

<div class="exercise"><h4>Ejercicio 2</h4><p>Inserta <code>0:(n-1)</code> después de intercambiar al azar el 5 % de los pares adyacentes: ¿la altura se acerca más al caso ordenado o al aleatorio? Luego inserta en el orden "primero la mediana, luego las medianas de las dos mitades, …". ¿Qué altura resulta y por qué es la mínima posible?</p></div>

## 4. Árboles balanceados

### 4.1 Por qué importa el balance y qué hace una rotación

Un árbol de búsqueda está <strong>balanceado</strong> cuando su altura es \(O(\log n)\) para cualquier secuencia de inserciones y eliminaciones, y es <em>autobalanceado</em> cuando restablece el balance después de cada operación con una cantidad de trabajo pequeña y local, de modo que toda operación es logarítmica en el peor caso y no solo en promedio. La herramienta para restablecerlo es la <strong>rotación</strong>: una reorganización de un nodo y uno de sus hijos que cambia en uno las alturas de los dos lados preservando la secuencia inorden y, por lo tanto, el invariante. En la Figura 3, una rotación a la derecha en torno a \(y\) con hijo izquierdo \(x\) convierte a \(x\) en el padre y a \(y\) en su hijo derecho, y mueve el subárbol \(B\) (claves entre \(x\) e \(y\)) de la derecha de \(x\) a la izquierda de \(y\), con un costo de un número constante de cambios de referencias.



In [ ]:
# Figura 3: una rotación a la derecha en torno a y. El inorden A x B y C se preserva; las alturas de los lados cambian en uno.
draw_rot <- function(top, child, side, main) {
  tx <- 0; ty <- 1; cx <- if (side == "left") -1 else 1; cy <- 0
  plot.new(); plot.window(xlim = c(-2.3, 2.3), ylim = c(-1.6, 1.6)); segments(tx, ty, cx, cy, col = GREY)
  tri <- function(ax, ay, lab) { polygon(c(ax - 0.35, ax + 0.35, ax), c(ay - 0.4, ay - 0.4, ay + 0.4), col = "#e3f1e3"); text(ax, ay - 0.1, lab, cex = 0.8) }
  if (side == "left") { segments(cx, cy, cx - 0.7, -1, col = GREY); segments(cx, cy, cx + 0.3, -1, col = GREY); segments(tx, ty, 1.2, -0.2, col = GREY)
    tri(cx - 0.7, -1, "A"); tri(cx + 0.3, -1, "B"); tri(1.2, -0.6, "C") }
  else { segments(tx, ty, -1.2, -0.2, col = GREY); segments(cx, cy, cx - 0.3, -1, col = GREY); segments(cx, cy, cx + 0.7, -1, col = GREY)
    tri(-1.2, -0.6, "A"); tri(cx - 0.3, -1, "B"); tri(cx + 0.7, -1, "C") }
  points(c(tx, cx), c(ty, cy), pch = 21, bg = "#dfe7f1", cex = 3.6); text(c(tx, cx), c(ty, cy), c(top, child), cex = 0.9)
  title(main = main, cex.main = 0.85)
}
par(mfrow = c(1, 2), mar = c(1, 1, 2, 1))
draw_rot("y", "x", "left", "before: y is the root, left-heavy")
draw_rot("x", "y", "right", "after a right rotation about y: x is the root")
mtext("in-order unchanged:  A  x  B  y  C", side = 1, outer = TRUE, line = -1.4, cex = 0.8, font = 3)



<p class="caption"><b>Figura 3.</b> Una rotación a la derecha en torno a \(y\): el subárbol \(A\) sube un nivel, \(C\) baja uno, \(B\) cambia de padre y la secuencia inorden \(A, x, B, y, C\) no cambia. La rotación a la izquierda es la imagen especular; una rotación doble resuelve el caso en que el subárbol pesado es \(B\).</p>

### 4.2 Árboles AVL

El primer árbol autobalanceado, publicado en 1962 por Adelson-Velskii y Landis, mantiene la <strong>condición AVL</strong>: en cada nodo, las alturas de los dos subárboles difieren a lo sumo en uno.<sup class="c"><a href="#r3">[3]</a></sup> Tras una inserción, el algoritmo recorre de vuelta el camino hacia arriba actualizando un <em>factor de balance</em> almacenado; en el primer nodo donde este llega a \(\pm 2\), una sola rotación simple o doble restablece la condición. La altura es a lo sumo de aproximadamente \(1.44\log_2(n+2)\), de modo que toda operación es \(O(\log n)\) en el peor caso; Knuth deriva la cota mediante los números de Fibonacci.<sup class="c"><a href="#r2">[2]</a></sup>

### 4.3 Árboles rojo–negro

Un <strong>árbol rojo–negro</strong> relaja la condición AVL a cambio de menos rotaciones: cada nodo lleva un bit de color, la raíz es negra, un nodo rojo no tiene hijos rojos y todo camino desde un nodo hasta las hojas vacías bajo él pasa por el mismo número de nodos negros. Así, ningún camino de la raíz a una hoja es más del doble de largo que otro, y la altura es a lo sumo \(2\log_2(n+1)\). Bayer introdujo la estructura en 1972 como <em>árbol B binario simétrico</em> (symmetric binary B-tree)<sup class="c"><a href="#r4">[4]</a></sup> y Guibas y Sedgewick le dieron la formulación con colores en 1978.<sup class="c"><a href="#r5">[5]</a></sup> Es la implementación habitual de los mapas ordenados (C++ <code>std::map</code>, Java <code>TreeMap</code>); los environments y las listas con nombre de R son tablas hash, más rápidas para la búsqueda exacta pero incapaces de responder consultas por rango.

### 4.4 Árboles B en bases de datos

En disco, el costo que importa es el número de bloques leídos, y un nodo binario desperdicia un bloque en una sola clave. El <strong>árbol B</strong> de Bayer y McCreight almacena cientos de claves por nodo, dimensionado a una página de disco, con todas las hojas a la misma profundidad y todos los nodos al menos medio llenos.<sup class="c"><a href="#r6">[6]</a></sup> Con un factor de ramificación de 500, caben 125 millones de claves en altura 3. Todo índice de una base de datos relacional usa una variante del árbol B; cuando el sistema de un hospital recupera un registro por su identificador en milisegundos, está recorriendo uno.

### 4.5 Experimento: costo de búsqueda en un árbol degenerado frente a uno balanceado

Implementar el rebalanceo AVL va más allá de la unidad, pero su <em>efecto</em> es fácil de mostrar: las mismas claves insertadas en orden dan una cadena, e insertadas con la mediana primero (la solución del Ejercicio 2) dan un árbol perfecto. La celda inserta 1,023 claves de ambas formas y cuenta las comparaciones de 2,000 búsquedas aleatorias.



In [ ]:
# Costo de búsqueda: 1,023 claves insertadas como cadena (en orden) frente a como árbol perfecto (mediana primero), 2,000 búsquedas en cada caso.
median_first <- function(keys) {                 # orden de inserción que construye un BST perfecto a partir de un vector ordenado
  if (!length(keys)) return(integer(0))
  m <- (length(keys) + 1) %/% 2
  c(keys[m], median_first(keys[seq_len(m - 1)]), median_first(keys[(m + 1):length(keys)][seq_len(length(keys) - m)]))
}
keys <- 0:1022                                    # 1023 = 2^10 - 1 -> es posible un árbol perfecto de altura 9
chain <- new_bst(); for (k in keys)                bst_insert(chain, k)
balanced <- new_bst(); for (k in median_first(keys)) bst_insert(balanced, k)
set.seed(1); queries <- sample(0:1022, 2000, replace = TRUE)
out <- do.call(rbind, lapply(list(c("degenerate (sorted insertion)", "chain"), c("balanced (median-first)", "balanced")),
  function(p) { t <- get(p[2]); t$comparisons <- 0L; for (q in queries) bst_search(t, q)
    data.frame(tree = p[1], height = tree_height(t$root), comparisons_per_search = round(t$comparisons / length(queries), 1),
               worst_case_h_plus_1 = tree_height(t$root) + 1) }))
print(out, row.names = FALSE)



El árbol balanceado tiene altura 9, la mínima para 1,023 claves, y responde una búsqueda con unas nueve comparaciones en promedio; la cadena tiene altura 1,022 y necesita unas quinientas para las mismas claves. Un árbol autobalanceado paga una pequeña constante por inserción para mantener logarítmica la segunda columna. En la segunda mitad de la unidad, los árboles toman su forma a partir de los datos y de los límites de profundidad, no de rotaciones, pero el costo de una predicción sigue siendo la profundidad de la hoja alcanzada, y un árbol profundo sigue siendo lento y —como veremos— demasiado específico.

<div class="box bridge"><h4>Puente con la bioinformática</h4><p>Un <em>árbol de sufijos</em> indexa un genoma de modo que cualquier subcadena se encuentra en un tiempo proporcional a su longitud; el <em>árbol k-d</em> y el ball tree aceleran la búsqueda de vecinos más cercanos de la Unidad 3; el <em>árbol filogenético</em> de UPGMA o de neighbour-joining es un árbol binario cuyo recorrido en postorden calcula los estados ancestrales; el dendrograma de agrupamiento (clustering) de la Unidad 5 es un árbol binario construido desde las hojas hacia arriba.</p></div>

## 5. Ejemplo resuelto a mano: construcción de un árbol de búsqueda

Inserta <strong>50, 30, 70, 20, 40, 60, 80</strong>, en este orden, en un BST vacío ("el menor va a la izquierda, el mayor a la derecha"):

| Paso | Clave | Comparaciones | Ubicación |
|---|---|---|---|
| 1 | 50 | — | raíz |
| 2 | 30 | 30 < 50 → izquierda | hijo izquierdo de 50 |
| 3 | 70 | 70 > 50 → derecha | hijo derecho de 50 |
| 4 | 20 | 20 < 50 → izquierda; 20 < 30 → izquierda | hijo izquierdo de 30 |
| 5 | 40 | 40 < 50 → izquierda; 40 > 30 → derecha | hijo derecho de 30 |
| 6 | 60 | 60 > 50 → derecha; 60 < 70 → izquierda | hijo izquierdo de 70 |
| 7 | 80 | 80 > 50 → derecha; 80 > 70 → derecha | hijo derecho de 70 |

```
            50
          /    \
        30      70
       /  \    /  \
     20   40  60   80
```

El árbol tiene 7 nodos y altura 2, y es perfecto: este orden es el orden con la mediana primero de la Sección 4.5. Sus recorridos: **preorden** 50, 30, 20, 40, 70, 60, 80; **inorden** 20, 30, 40, 50, 60, 70, 80 (ordenado, como exige el invariante); **postorden** 20, 40, 30, 60, 80, 70, 50; **por niveles** 50, 30, 70, 20, 40, 60, 80. Buscar 60 sigue el camino 50 → 70 → 60 (tres comparaciones); buscar 65 sigue el mismo camino, no encuentra hijo derecho de 60 e informa su ausencia tras tres comparaciones. Las mismas claves insertadas en orden dan una cadena de altura 6 en la que la búsqueda de 80 toma siete comparaciones. La celda confirma cada afirmación y dibuja ambos árboles.



In [ ]:
# Confirmación en código del ejemplo resuelto a mano de la Sección 5; luego se dibujan ambos árboles a partir de la estructura.
hand <- new_bst(); for (k in c(50, 30, 70, 20, 40, 60, 80)) bst_insert(hand, k)
cat("nodes:", tree_count(hand$root), "| height:", tree_height(hand$root), "| leaves:", paste(tree_leaves(hand$root), collapse = " "), "\n")
for (nm in c("pre-order", "in-order", "post-order", "level-order"))
  cat(sprintf("%-12s %s\n", nm, paste(switch(nm, "pre-order" = preorder, "in-order" = inorder, "post-order" = postorder, levelorder)(hand$root), collapse = " ")))
for (q in c(60, 65)) { s <- bst_search(hand, q); cat(sprintf("search %d: %s (%d comparisons)\n", q, if (s$found) "found" else "absent", length(s$path))) }
chain7 <- new_bst(); for (k in sort(c(50, 30, 70, 20, 40, 60, 80))) bst_insert(chain7, k)
cat("same keys inserted sorted -> height", tree_height(chain7$root), "| search 80 takes", length(bst_search(chain7, 80)$path), "comparisons\n")

draw_bst <- function(root, main) {
  ord <- inorder(root); pos <- new.env(); hgt <- tree_height(root)
  place <- function(nd, d) { if (is.null(nd)) return(); x <- match(nd$val, ord); assign(as.character(nd$val), c(x, -d), pos)
    for (ch in list(nd$left, nd$right)) if (!is.null(ch)) segments(x, -d, match(ch$val, ord), -d - 1, col = GREY)
    place(nd$left, d + 1); place(nd$right, d + 1) }
  plot.new(); plot.window(xlim = c(0.5, length(ord) + 0.5), ylim = c(-hgt - 0.6, 0.6)); place(root, 0)
  for (k in ord) { p <- get(as.character(k), pos); points(p[1], p[2], pch = 21, bg = "#dfe7f1", cex = 2.8); text(p[1], p[2], k, cex = 0.7) }
  title(main = main, cex.main = 0.85)
}
par(mfrow = c(1, 2), mar = c(1, 1, 2, 1))
draw_bst(hand$root, "inserted 50,30,70,20,40,60,80 (height 2)")
draw_bst(chain7$root, "same keys inserted sorted (height 6)")



<p class="caption"><b>Figura 4.</b> Los dos árboles del ejemplo resuelto, dibujados a partir de la estructura de datos: la posición horizontal es el rango en inorden y la posición vertical, la profundidad. Mismas claves, misma secuencia inorden, alturas 2 y 6.</p>

El código reproduce exactamente la traza a mano; deberías ser capaz de realizarla en papel para cualquier lista corta de claves.

## 6. De los árboles binarios a los árboles de decisión

### 6.1 Un árbol de decisión es un árbol binario sobre el espacio de características

El conjunto de datos recurrente del curso son los datos de <strong>cáncer de mama de Wisconsin</strong>. El paquete <code>mlbench</code> incluye la versión original de Wolberg y colaboradores: 699 aspirados con aguja fina, nueve características citológicas puntuadas cada una por un patólogo de 1 a 10, etiquetados como benignos o malignos.<sup class="c"><a href="#r28">[28]</a>,<a href="#r4">[4]</a></sup> Es el sustituto en R del conjunto de datos usado en toda la edición en Python. Un clasificador de <strong>árbol de decisión</strong> es un árbol binario en el que cada nodo interno contiene una prueba "¿es la característica \(j\) ≤ umbral \(t\)?", se toma el hijo izquierdo cuando la respuesta es sí, y cada hoja contiene una clase. Clasificar a un paciente es una búsqueda en el sentido de la Sección 3, guiada por las características del paciente, en la que el camino mismo es la respuesta. Geométricamente, cada prueba es un hiperplano perpendicular a un eje, de modo que las hojas dividen el espacio de características en cajas alineadas con los ejes. La celda carga y depura los datos, ajusta un árbol de profundidad 2 a la parte de entrenamiento de una partición 70/30 (usada en el resto de la unidad, siguiendo el Suplemento S1), lo dibuja e inspecciona la representación interna de <code>rpart</code>.



In [ ]:
# Cargar y depurar los datos; un árbol de profundidad 2; su representación interna como data frame.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # eliminar la columna Id
bc <- bc[complete.cases(bc), ]                             # eliminar las 16 filas con valores faltantes
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))   # ordinal -> numérica
set.seed(1)
idx <- sample(nrow(bc), round(0.7 * nrow(bc)))
tr <- bc[idx, ]; te <- bc[-idx, ]
cat(sprintf("%d complete cases; training %d, test %d; classes: %s\n", nrow(bc), nrow(tr), nrow(te),
            paste(names(table(bc$Class)), table(bc$Class), collapse = ", ")))

ctl <- function(d) rpart.control(maxdepth = d, cp = 0, minsplit = 2, minbucket = 1, xval = 0)
set.seed(1); tree2 <- rpart(Class ~ ., tr, method = "class", control = ctl(2))
rpart.plot(tree2, type = 2, extra = 104, box.palette = list("#fbe3d3", "#dfe7f1"), main = "A depth-2 tree (training part)")
print(tree2$frame[, c("var", "n", "dev", "yval")])
cat("node numbers (row names):", paste(rownames(tree2$frame), collapse = " "), "\n")



<p class="caption"><b>Figura 5.</b> Un árbol de profundidad 2 aprendido a partir de 478 casos de entrenamiento. Cada caja muestra la clase mayoritaria, las proporciones de clase y la fracción de casos de entrenamiento que llegan al nodo; un nodo interno muestra además su prueba. Un camino de la raíz a una hoja es una regla: la primera pregunta se refiere a la forma celular; la siguiente, al tamaño celular o al aspecto de los núcleos desnudos.</p>

El data frame impreso demuestra la afirmación con que comenzó la sección: <code>rpart</code> almacena el árbol como una tabla indexada por nodo, en la que la variable especial <code>&lt;leaf&gt;</code> marca una hoja. El árbol tiene 7 nodos, el conteo del árbol perfecto de la Sección 2.5, y los nombres de fila —los números de nodo— son 1, 2, 4, 5, 3, 6, 7: exactamente la regla del vector de la Sección 2.3, con los hijos del nodo \(i\) ubicados en \(2i\) y \(2i+1\). Todo lo que imprimen <code>rpart.plot</code> y el texto de reglas es un recorrido de esta tabla.

### 6.2 Impureza: ¿qué tan buena es una división?

Aprender un árbol significa elegir las pruebas. La idea común a las dos tradiciones que inventaron el método —el CART estadístico de Breiman, Friedman, Olshen y Stone<sup class="c"><a href="#r7">[7]</a></sup> y los ID3 y C4.5 de Quinlan<sup class="c"><a href="#r8">[8]</a>,<a href="#r9">[9]</a></sup>— es que una buena prueba envía las dos clases a lados distintos. La <strong>impureza</strong> de un nodo vuelve esto numérico: vale 0 cuando todos los casos comparten una clase y es máxima cuando las clases están mezcladas en partes iguales. Para proporciones de clase \(p_1, \dots, p_K\):

\[ \text{Gini: } G = 1 - \sum_k p_k^2, \qquad \text{Entropy: } H = -\sum_k p_k \log_2 p_k, \qquad \text{Misclassification: } E = 1 - \max_k p_k . \]

Gini es la probabilidad de que dos casos extraídos del nodo pertenezcan a clases distintas (la opción por defecto de CART, y de <code>rpart</code>); la entropía es el número esperado de bits para codificar la clase de un caso (ID3); el error de clasificación es el error de la predicción mayoritaria. Para dos clases, \(G = 2p(1-p)\), \(H = -p\log_2 p - (1-p)\log_2(1-p)\), \(E = \min(p, 1-p)\).

La <strong>calidad de una división</strong> es la disminución de la impureza, ponderada por el número de casos de cada lado:

\[ \Delta I = I - \left( \frac{n_L}{n} I_L + \frac{n_R}{n} I_R \right), \]

llamada <strong>ganancia de información</strong> cuando \(I\) es la entropía. La división elegida maximiza \(\Delta I\).



In [ ]:
# Las tres medidas de impureza para un nodo de dos clases en función de la proporción p de una clase.
p <- seq(0.001, 0.999, length.out = 400)
gini <- 2 * p * (1 - p)
entropy <- -(p * log2(p) + (1 - p) * log2(1 - p))
miscl <- pmin(p, 1 - p)
plot(p, entropy, type = "l", col = BLUE, lwd = 2, ylim = c(0, 1.05),
     xlab = "proportion p of one class at the node", ylab = "impurity", main = "Impurity of a two-class node")
lines(p, gini, col = ORANGE, lwd = 2); lines(p, miscl, col = GREEN, lwd = 2); lines(p, entropy / 2, col = BLUE, lty = 3)
legend("topright", bty = "n", cex = 0.8, lwd = 2, col = c(BLUE, ORANGE, GREEN, BLUE), lty = c(1, 1, 1, 3),
       legend = c("entropy", "Gini = 2p(1-p)", "misclassification = min(p,1-p)", "entropy / 2"))
cat(sprintf("at p=0.5: Gini %.3f, entropy %.3f, misclassification %.3f\n", 0.5, 1.0, 0.5))
cat(sprintf("at p=0.9: Gini %.3f, entropy %.3f, misclassification %.3f\n", 2 * 0.9 * 0.1, -(0.9 * log2(0.9) + 0.1 * log2(0.1)), 0.1))



<p class="caption"><b>Figura 6.</b> Las tres medidas de impureza para un nodo de dos clases: todas se anulan en un nodo puro y alcanzan su máximo en \(p = 0.5\). Gini y entropía son estrictamente cóncavas y tienen casi la misma forma (punteado: entropía reescalada); el error de clasificación es lineal por tramos.</p>

La forma importa más que los valores. Como Gini y entropía son <em>estrictamente cóncavas</em>, cualquier división que deje a los hijos menos mezclados las hace disminuir aunque ningún hijo cambie su clase mayoritaria; el error de clasificación puede quedar sin cambio ante una división útil cuando ambos hijos conservan la clase mayoritaria del padre (un padre 60 : 40 dividido en 30 : 10 y 30 : 30 mantiene \(E = 0.4 \times 0.25 + 0.6 \times 0.5 = 0.40\), el valor del padre, mientras que Gini cae de 0.48 a \(0.4 \times 0.375 + 0.6 \times 0.5 = 0.45\)). Por eso CART y C4.5 hacen crecer el árbol con Gini o entropía y usan el error de clasificación solo para <em>evaluar</em> el árbol terminado.<sup class="c"><a href="#r7">[7]</a>,<a href="#r23">[23]</a></sup>

### 6.3 El algoritmo recursivo voraz, con una división calculada a mano

CART hace crecer un árbol de clasificación así:

1. Comienza con todos los casos de entrenamiento en la raíz.
2. Para cada característica \(j\) y cada umbral \(t\) entre valores ordenados consecutivos de \(x_j\) en el nodo, calcula \(\Delta I\) para la división "\(x_j \le t\)".
3. Toma la división con el mayor \(\Delta I\) y envía los casos a dos hijos.
4. Repite recursivamente en cada hijo hasta que se cumpla una regla de detención: nodo puro, muy pocos casos, profundidad máxima, o ninguna división disminuye la impureza.
5. Etiqueta cada hoja con su clase mayoritaria (o con sus proporciones de clase).

El algoritmo es <strong>voraz</strong> (greedy): elige la mejor división en el momento y nunca la revisa. No tiene por qué encontrar el árbol más pequeño ni el más exacto —encontrar un árbol óptimo es NP-completo<sup class="c"><a href="#r10">[10]</a></sup>—, pero un nivel cuesta solo \(O(p\, n \log n)\).

<strong>A mano.</strong> Diez pacientes, cinco malignos (M) y cinco benignos (B); una prueba envía seis a la izquierda (5 B, 1 M) y cuatro a la derecha (4 M).

- Padre: \(G = 1 - (0.5^2 + 0.5^2) = 0.5\); \(H = 1\) bit.
- Izquierdo (5 B, 1 M): \(G_L = 1 - (5/6)^2 - (1/6)^2 = 10/36 \approx 0.278\); \(H_L = -\tfrac56\log_2\tfrac56 - \tfrac16\log_2\tfrac16 \approx 0.650\) bits.
- Derecho (4 M): \(G_R = H_R = 0\).
- Hijos ponderados: \(0.6 \times 0.278 = 0.167\); \(0.6 \times 0.650 = 0.390\) bits.
- Disminución: \(\Delta G = 0.333\); ganancia de información \(0.610\) bits.

Una prueba competidora que envía 3 B + 2 M a la izquierda y 2 B + 3 M a la derecha da \(G_L = G_R = 0.48\) y \(\Delta G = 0.02\). La celda confirma la aritmética.



In [ ]:
# Funciones de impureza y la división calculada a mano de la Sección 6.3.
gini_index <- function(labels) { pk <- table(labels) / length(labels); 1 - sum(pk^2) }
entropy_bits <- function(labels) { pk <- table(labels) / length(labels); pk <- pk[pk > 0]; -sum(pk * log2(pk)) }
split_quality <- function(yL, yR, impurity = gini_index) {
  yall <- c(yL, yR); n <- length(yall); child <- (length(yL) / n) * impurity(yL) + (length(yR) / n) * impurity(yR)
  c(decrease = impurity(yall) - child, child = child)
}
left <- rep(c("B", "M"), c(5, 1)); right <- rep("M", 4)           # 5B+1M a la izquierda, 4M a la derecha
g <- split_quality(left, right, gini_index); h <- split_quality(left, right, entropy_bits)
cat(sprintf("parent Gini %.3f; left %.3f; right %.3f; weighted child %.3f -> decrease %.3f\n",
    gini_index(c(left, right)), gini_index(left), gini_index(right), g["child"], g["decrease"]))
cat(sprintf("parent entropy %.3f bits; left %.3f; weighted child %.3f -> information gain %.3f bits\n",
    entropy_bits(c(left, right)), entropy_bits(left), h["child"], h["decrease"]))
cat(sprintf("competing split (3B+2M | 2B+3M): decrease in Gini %.3f\n",
    split_quality(rep(c("B","M"), c(3,2)), rep(c("B","M"), c(2,3)))["decrease"]))



Se reproducen todos los números del cálculo a mano. Este procedimiento —impurezas de los hijos, ponderadas por tamaño y restadas de la del padre— es todo el ciclo interno de CART.

### 6.4 Una división de una característica real, por búsqueda exhaustiva y con rpart

El paso 2, para una característica, es un barrido: se ordenan los valores distintos en el nodo, se toma como umbral candidato el punto medio entre valores consecutivos y se calcula la impureza ponderada de los hijos para cada uno. La celda barre <em>cell size</em> (tamaño celular) sobre los 478 casos de entrenamiento y luego le pide a <code>rpart</code> con <code>maxdepth = 1</code> —un <em>decision stump</em>— que haga lo mismo, primero solo con esa característica y luego con las nueve características.



In [ ]:
# Barrido exhaustivo de los umbrales de 'Cell.size' (el ciclo interno de CART) y luego un árbol de profundidad 1.
wgini <- function(f, t, y) { l <- y[f <= t]; r <- y[f > t]; if (!length(l) || !length(r)) return(NA)
  (length(l) * gini_index(l) + length(r) * gini_index(r)) / length(y) }
f <- tr$Cell.size; y <- tr$Class
vals <- sort(unique(f)); cand <- (head(vals, -1) + tail(vals, -1)) / 2
wg <- sapply(cand, wgini, f = f, y = y); best <- cand[which.min(wg)]
cat(sprintf("%d candidate thresholds; parent Gini %.4f\n", length(cand), gini_index(y)))
cat(sprintf("best Cell.size <= %.1f: weighted child Gini %.4f, decrease %.4f\n", best, min(wg), gini_index(y) - min(wg)))
set.seed(1); stump1 <- rpart(Class ~ Cell.size, tr, method = "class", control = rpart.control(maxdepth = 1, cp = 0, xval = 0))
set.seed(1); stumpA <- rpart(Class ~ ., tr, method = "class", control = rpart.control(maxdepth = 1, cp = 0, xval = 0))
cat(sprintf("rpart stump on Cell.size only : split %.1f; test accuracy %.3f\n",
    stump1$splits[1, "index"], mean(predict(stump1, te, type = "class") == te$Class)))
cat(sprintf("rpart stump on all 9 features : splits on '%s'; test accuracy %.3f\n",
    rownames(stumpA$splits)[1], mean(predict(stumpA, te, type = "class") == te$Class)))
plot(cand, wg, type = "l", col = BLUE, lwd = 1.5, xlab = "candidate threshold on 'Cell.size'",
     ylab = "weighted Gini of the two children", main = "Scanning every threshold of one feature (CART's inner loop)")
abline(v = best, col = ORANGE, lty = 2); points(best, min(wg), pch = 19, col = ORANGE)



<p class="caption"><b>Figura 7.</b> Gini ponderado de los dos hijos en función del umbral sobre <em>cell size</em>, para los puntos medios candidatos. El mínimo está en 3.5: los tumores cuyo puntaje de tamaño celular lo supera son abrumadoramente malignos.</p>

El barrido encuentra el umbral 3.5 con Gini ponderado 0.125, frente a 0.460 en el padre, y el decision stump de <code>rpart</code> sobre la misma característica encuentra <em>exactamente el mismo umbral</em>: su algoritmo es el barrido que acabamos de escribir. Con las nueve características, el decision stump prefiere <em>cell shape</em> (forma celular), casi un duplicado del tamaño celular; cuál de las dos gana es cuestión de unos pocos casos de entrenamiento, un primer indicio de la inestabilidad de la Sección 6.10. Una sola pregunta ya clasifica correctamente cerca de nueve de cada diez casos de prueba.

<div class="box warn"><h4>Por qué solo puntos medios, y por qué los árboles no necesitan escalamiento</h4><p>Cualquier umbral estrictamente entre dos valores ordenados consecutivos da la misma división, así que basta un representante por intervalo. De ello se sigue que un árbol es <strong>invariante frente a cualquier transformación monótona de una característica</strong>: los logaritmos, la estandarización o un cambio de unidades alteran los valores de los umbrales pero no las divisiones. Por eso los árboles no necesitan escalar las características, a diferencia de k-NN y SVM en la Unidad 3.</p></div>

### 6.5 Crecimiento del árbol completo y lectura de sus reglas

Si se deja correr hasta que cada hoja sea pura, el algoritmo clasifica correctamente todos los casos de entrenamiento. La celda hace crecer ese árbol, informa su tamaño y exactitudes, e imprime las reglas de un árbol de profundidad 3.



In [ ]:
# El árbol sin podar: perfecto en los datos de entrenamiento, peor en los de prueba. Reglas como recorrido en preorden.
set.seed(1); full <- rpart(Class ~ ., tr, method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
nleaf <- function(f) sum(f$frame$var == "<leaf>")
cat(sprintf("unpruned tree: %d leaves, %d nodes; train acc %.3f, test acc %.3f\n",
    nleaf(full), nrow(full$frame), mean(predict(full, tr, type = "class") == tr$Class), mean(predict(full, te, type = "class") == te$Class)))
set.seed(1); tree3 <- rpart(Class ~ ., tr, method = "class", control = ctl(3))
cat(sprintf("depth-3 tree: %d leaves; train acc %.3f, test acc %.3f\n\n",
    nleaf(tree3), mean(predict(tree3, tr, type = "class") == tr$Class), mean(predict(tree3, te, type = "class") == te$Class)))
print(tree3)                                              # las reglas, como un recorrido en preorden con sangría
rpart.plot(tree3, type = 2, extra = 8, box.palette = list("#fbe3d3", "#dfe7f1"), main = "The depth-3 tree drawn")



<p class="caption"><b>Figura 8.</b> El árbol de profundidad 3, dibujado. Cualquier camino de la raíz a una hoja es una regla SI–ENTONCES: la primera pregunta se refiere a la forma celular; las siguientes, al tamaño celular, a los núcleos desnudos o al grosor del grupo celular (clump thickness). Las hojas a las que llegan uno o dos casos de entrenamiento son las que con mayor probabilidad codifican ruido.</p>

El árbol sin podar alcanza 100 % de exactitud de entrenamiento, como debe ser, con 23 hojas y profundidad 9; en los 205 casos de prueba acierta en el 94.1 %. El árbol de profundidad 3, con muchas menos hojas, lo iguala en los casos de prueba: las divisiones adicionales aprendieron los accidentes de los 478 casos de entrenamiento, no el patrón. Esto es <strong>sobreajuste</strong>, el tema del Suplemento S1, y la profundidad es la perilla que lo controla. El árbol impreso es un recorrido en preorden en el que la sangría codifica la profundidad y cada hoja muestra sus conteos por clase.

### 6.6 Profundidad frente a sobreajuste

El experimento hace crecer árboles de profundidad máxima 1 a 8 y registra la exactitud de entrenamiento, la exactitud por validación cruzada de 5 particiones dentro de la parte de entrenamiento (la estimación honesta de S1) y la exactitud de prueba, que se muestra solo a modo ilustrativo y nunca se usa para elegir la profundidad.



In [ ]:
# Exactitud de entrenamiento, por validación cruzada y de prueba frente a la profundidad máxima del árbol.
acc <- function(p, y) mean(p == y)
kfold <- function(d, k = 5) { set.seed(1); fold <- sample(rep(1:k, length.out = nrow(tr)))
  sapply(1:k, function(i) { fit <- rpart(Class ~ ., tr[fold != i, ], method = "class", control = ctl(d))
    acc(predict(fit, tr[fold == i, ], type = "class"), tr$Class[fold == i]) }) }
curve <- do.call(rbind, lapply(1:8, function(d) { set.seed(1); ft <- rpart(Class ~ ., tr, method = "class", control = ctl(d)); cv <- kfold(d)
  data.frame(depth = d, train = acc(predict(ft, tr, type = "class"), tr$Class), cv = mean(cv), cvsd = sd(cv),
             test = acc(predict(ft, te, type = "class"), te$Class), leaves = nleaf(ft)) }))
best_d <- curve$depth[which.max(curve$cv)]
plot(curve$depth, curve$train, type = "b", pch = 19, col = GREY, ylim = c(0.86, 1.01),
     xlab = "maximum depth of the tree", ylab = "accuracy", main = "Depth controls overfitting")
arrows(curve$depth, curve$cv - curve$cvsd, curve$depth, curve$cv + curve$cvsd, length = 0.02, angle = 90, code = 3, col = BLUE)
lines(curve$depth, curve$cv, type = "b", pch = 15, col = BLUE); lines(curve$depth, curve$test, type = "b", pch = 17, lty = 2, col = ORANGE)
abline(v = best_d, col = BLUE, lty = 3)
legend("bottomright", bty = "n", cex = 0.8, col = c(GREY, BLUE, ORANGE), pch = c(19, 15, 17), lty = c(1, 1, 2),
       legend = c("training", "5-fold CV (training part)", "test (not used for selection)"))
cat("depth with the best cross-validated accuracy:", best_d, "\n"); print(round(curve, 3), row.names = FALSE)



<p class="caption"><b>Figura 9.</b> Exactitud frente a profundidad máxima. La exactitud de entrenamiento sube hasta 1.0; la exactitud por validación cruzada (una desviación estándar entre particiones) alcanza su máximo en torno a la profundidad 3 y luego disminuye levemente; la exactitud de prueba se estabiliza. Las diferencias de uno o dos puntos entre profundidades vecinas caen dentro de las barras de error.</p>

Más allá de cierta complejidad, la capacidad adicional solo compra exactitud de entrenamiento y nada más. El óptimo por validación cruzada es la profundidad 3 (0.964), pero la desviación estándar entre particiones es de dos puntos, de modo que la afirmación honesta es "una profundidad de entre 2 y 4, aproximadamente, es tan buena como cualquier otra". Elegir la profundidad según la curva naranja de prueba sería el error del Suplemento S1: usar el conjunto de prueba para seleccionar el modelo.

### 6.7 Poda por costo-complejidad

Un límite de profundidad corta todas las ramas al mismo nivel, aunque algunas merecerían ser más profundas. La <strong>poda por costo-complejidad</strong> de CART hace crecer el árbol completo y luego elimina las ramas que menos aportan al ajuste en relación con su tamaño: para un árbol \(T\) con \(|T|\) hojas y error de entrenamiento \(R(T)\),

\[ R_\alpha(T) = R(T) + \alpha\,|T| , \]

donde \(\alpha \ge 0\) es el precio de una hoja. A medida que \(\alpha\) crece, el subárbol óptimo pierde sus ramas más débiles una a una; los subárboles óptimos están anidados, de modo que toda la secuencia es barata de calcular, y \(\alpha\) se elige por validación cruzada.<sup class="c"><a href="#r7">[7]</a>,<a href="#r23">[23]</a></sup> En <code>rpart</code> el parámetro es <code>cp</code>, un \(\alpha\) reescalado, y la función aplica validación cruzada a cada subárbol de forma automática e informa el resultado en una <em>tabla de complejidad</em>.<sup class="c"><a href="#r34">[34]</a></sup>



In [ ]:
# La propia tabla de complejidad de rpart, con validación cruzada de 10 particiones, es el camino de poda; elegir el subárbol con menor error de VC.
set.seed(1); tcp <- rpart(Class ~ ., tr, method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 10))
print(round(tcp$cptable, 4))
cp_opt <- tcp$cptable[which.min(tcp$cptable[, "xerror"]), "CP"]
pruned <- prune(tcp, cp = cp_opt * 1.0001)
cat(sprintf("best CV subtree: cp %.4f, %d leaves, test accuracy %.3f\n",
    cp_opt, nleaf(pruned), acc(predict(pruned, te, type = "class"), te$Class)))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(tcp$cptable[, "nsplit"] + 1, tcp$cptable[, "rel error"], type = "b", pch = 19, col = GREY,
     xlab = "number of leaves", ylab = "error", main = "Training vs cross-validated error")
arrows(tcp$cptable[, "nsplit"] + 1, tcp$cptable[, "xerror"] - tcp$cptable[, "xstd"],
       tcp$cptable[, "nsplit"] + 1, tcp$cptable[, "xerror"] + tcp$cptable[, "xstd"], length = 0.02, angle = 90, code = 3, col = BLUE)
lines(tcp$cptable[, "nsplit"] + 1, tcp$cptable[, "xerror"], type = "b", pch = 15, col = BLUE)
legend("topright", bty = "n", cex = 0.8, col = c(GREY, BLUE), pch = c(19, 15), legend = c("training (rel. error)", "10-fold CV error"))
abline(v = nleaf(pruned), col = ORANGE, lty = 2)
rpart.plot(pruned, type = 2, extra = 104, box.palette = list("#fbe3d3", "#dfe7f1"), main = "The cross-validated best subtree")



<p class="caption"><b>Figura 10.</b> Izquierda: el error de entrenamiento cae de forma monótona con el número de hojas, pero el error por validación cruzada de 10 particiones (una desviación estándar) toca fondo con unas pocas hojas y luego sube: el sobreajuste de la Sección 6.6 bajo otra apariencia. Derecha: el mejor subárbol por validación cruzada, aquel cuyas reglas se le mostrarían a un clínico.</p>

El mejor subárbol tiene 5 hojas, un error por validación cruzada de alrededor de 0.10 y una exactitud de prueba de 0.946: tan bueno como el árbol de 23 hojas con una quinta parte de las hojas; las ramas adicionales estaban ajustando ruido. La regla más simple de "un error estándar", que toma el árbol más pequeño cuyo error está a no más de una desviación estándar del mínimo, podaría todavía más, hasta un árbol de 3 hojas. Este árbol podado es el árbol individual que se compara con el bosque en la Sección 7.

### 6.8 Importancia de las características y sus salvedades

Un árbol ofrece una medida natural de la contribución de cada característica: la disminución total de impureza de todas las divisiones sobre esa característica, ponderada por el tamaño del nodo; es la <strong>importancia basada en impureza</strong> (disminución media de la impureza, MDI, mean decrease in impurity).<sup class="c"><a href="#r7">[7]</a>,<a href="#r24">[24]</a></sup> Strobl y colaboradores documentaron dos salvedades.<sup class="c"><a href="#r16">[16]</a></sup> Está <em>sesgada hacia las características con muchos puntos de corte</em>: una característica que ofrece a la búsqueda voraz más umbrales candidatos tiene más oportunidades de encontrar una división espuriamente buena, de modo que incluso una característica de puro ruido con muchos valores recibe importancia. Y entre características <em>correlacionadas</em>, el crédito se reparte de manera arbitraria: se lo lleva cualquiera de las casi duplicadas que se elija primero. La celda muestra el primer sesgo en forma pura —cuatro características de ruido con distinto número de niveles y nada más— usando un bosque (Sección 7) para promediar la arbitrariedad de los árboles individuales.



In [ ]:
# Salvedad 1 de la importancia por impureza: cuatro características de PURO RUIDO con distinto número de umbrales candidatos.
set.seed(1); nn <- nrow(tr)
noise <- data.frame(Class = tr$Class, noise_cont = rnorm(nn), noise_20 = sample(0:19, nn, TRUE),
                    noise_4 = sample(0:3, nn, TRUE), noise_bin = sample(0:1, nn, TRUE))
set.seed(1); rf_null <- randomForest(Class ~ ., noise, ntree = 300)
cat("impurity importance (MeanDecreaseGini) when EVERY feature is noise (should be equal if unbiased):\n")
print(round(importance(rf_null)[, 1], 1))



Cuando todas las características son ruido, las importancias deberían ser iguales; en cambio, quedan ordenadas según el número de umbrales candidatos: alrededor de 104 para la característica continua frente a 9 para la binaria, sin información en ninguna de las dos. Las reglas: nunca interpretes la importancia por impureza como relevancia causal o marginal; compárala con la importancia por permutación (Sección 7.4); informa las características correlacionadas como grupo.<sup class="c"><a href="#r16">[16]</a>,<a href="#r17">[17]</a></sup>

### 6.9 Árboles de regresión

El mismo algoritmo ajusta una variable objetivo continua. En un <strong>árbol de regresión</strong>, cada hoja predice la <em>media</em> de los valores objetivo de entrenamiento que llegan a ella, la impureza es su <em>varianza</em> y la predicción es constante por tramos: una función escalonada en una dimensión. La edición en Python usó los datos de diabetes (442 pacientes, progresión de la enfermedad al cabo de un año);<sup class="c"><a href="#r29">[29]</a></sup> el sustituto en R son los datos de viviendas de Boston incluidos en <code>mlbench</code> (506 sectores censales, valor mediano de las viviendas a partir de trece covariables),<sup class="c"><a href="#r33">[33]</a>,<a href="#r4">[4]</a></sup> que muestran el comportamiento igual de bien.



In [ ]:
# Un árbol de regresión sobre los datos de Boston: R^2 en la parte de prueba y el ajuste escalonado a lo largo de una covariable.
data(BostonHousing, package = "mlbench"); bh <- BostonHousing
set.seed(1); bi <- sample(nrow(bh), round(0.7 * nrow(bh))); btr <- bh[bi, ]; bte <- bh[-bi, ]
r2 <- function(p, y) 1 - sum((y - p)^2) / sum((y - mean(y))^2)
set.seed(1); reg2  <- rpart(medv ~ ., btr, method = "anova", control = rpart.control(maxdepth = 2, cp = 0, xval = 0))
set.seed(1); regf  <- rpart(medv ~ ., btr, method = "anova", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
cat(sprintf("depth-2 tree splits on {%s}: test R^2 %.3f\n", paste(setdiff(unique(as.character(reg2$frame$var)), "<leaf>"), collapse = ", "), r2(predict(reg2, bte), bte$medv)))
cat(sprintf("unpruned tree (%d leaves): train R^2 %.3f, test R^2 %.3f\n", nleaf(regf), r2(predict(regf, btr), btr$medv), r2(predict(regf, bte), bte$medv)))
grid <- seq(min(bh$rm), max(bh$rm), length.out = 400)
plot(btr$rm, btr$medv, pch = 19, cex = 0.5, col = GREY, xlab = "average number of rooms (rm)", ylab = "median home value (medv)",
     main = "A regression tree predicts a piecewise-constant function")
for (dc in list(c(1, ORANGE), c(3, BLUE))) { d <- as.integer(dc[1])
  r <- rpart(medv ~ rm, btr, method = "anova", control = rpart.control(maxdepth = d, cp = 0, xval = 0))
  pr <- predict(r, data.frame(rm = grid)); lines(grid, pr, col = dc[2], lwd = 1.8, type = "s") }
legend("topleft", bty = "n", cex = 0.8, lwd = 1.8, col = c(ORANGE, BLUE), legend = c("tree on rm, depth 1", "tree on rm, depth 3"))



<p class="caption"><b>Figura 11.</b> Valor mediano de las viviendas frente al número medio de habitaciones para los sectores censales de entrenamiento, con árboles de regresión de profundidad 1 y 3 ajustados solo a esta característica: constantes dentro de cada hoja, con saltos en cada umbral.</p>

El árbol de profundidad 2 explica alrededor de un tercio de la varianza de los sectores no vistos, y el árbol sin podar —una hoja por cada pocos sectores— tiene un \(R^2\) de prueba <em>menor</em> que el árbol poco profundo pese a un ajuste perfecto en entrenamiento: de nuevo, sobreajuste. La función escalonada también muestra una limitación: un árbol representa una tendencia suave solo como una escalera y extrapola como una constante. Los bosques heredan ambas cosas, atenuadas.

### 6.10 Fortalezas y debilidades de un árbol individual

<strong>Fortalezas.</strong> No requiere escalar las características (Sección 6.4); admite características continuas y categóricas en un mismo modelo, y valores faltantes mediante surrogate splits (nativos en <code>rpart</code>); interpretabilidad: el modelo es una lista corta de reglas y cada predicción, un camino; entrenamiento rápido, \(O(p\,n\log n)\) por nivel, y predicción en \(O(\text{depth})\); las interacciones se capturan automáticamente.

<strong>Debilidades.</strong> <strong>Inestabilidad</strong>: el algoritmo es voraz y cada división se elige por un margen pequeño, de modo que un pequeño cambio en los datos de entrenamiento puede cambiar la raíz y todo lo que está debajo; es la propiedad que Breiman señaló al introducir el bagging.<sup class="c"><a href="#r11">[11]</a></sup> Y <strong>fronteras alineadas con los ejes</strong>: una frontera oblicua debe aproximarse con una escalera de muchas divisiones. Dos celdas demuestran cada una.



In [ ]:
# Debilidad 1, inestabilidad: árboles crecidos sobre dos muestras bootstrap de los mismos datos de entrenamiento.
set.seed(1); b1 <- sample(nn, nn, replace = TRUE); b2 <- sample(nn, nn, replace = TRUE)
cat(sprintf("bootstrap sample 1: %d distinct training cases of %d (%.1f%%)\n", length(unique(b1)), nn, 100 * length(unique(b1)) / nn))
t1 <- rpart(Class ~ ., tr[b1, ], method = "class", control = ctl(2)); t2b <- rpart(Class ~ ., tr[b2, ], method = "class", control = ctl(2))
cat("depth-2 tree, sample 1 splits (root first):", paste(setdiff(unique(as.character(t1$frame$var)), "<leaf>"), collapse = ", "), "\n")
cat("depth-2 tree, sample 2 splits (root first):", paste(setdiff(unique(as.character(t2b$frame$var)), "<leaf>"), collapse = ", "), "\n")
set.seed(1); f1 <- rpart(Class ~ ., tr[b1, ], method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
set.seed(1); f2 <- rpart(Class ~ ., tr[b2, ], method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
cat(sprintf("two unpruned trees from the two samples disagree on %.1f%% of the %d test cases\n",
    100 * mean(predict(f1, te, type = "class") != predict(f2, te, type = "class")), nrow(te)))



Cada muestra bootstrap contiene alrededor del 63 % de los casos de entrenamiento distintos (la Sección 7.3 explica la cifra), así que las dos muestras difieren solo en cuáles de los mismos 478 pacientes se repiten. Aquí los dos árboles comparten la misma raíz —la forma celular y el tamaño celular son duplicados tan cercanos que la elección apenas se disputa—, pero difieren en los umbrales y en las características bajo la raíz, y dos árboles sin podar obtenidos de las dos muestras discrepan en un porcentaje de varias unidades de los casos de prueba. Para una regla clínica esto es incómodo, y es la propiedad que la Sección 7 convierte en una ventaja.



In [ ]:
# Debilidad 2, fronteras alineadas con los ejes: un árbol sobre dos características divide el plano en rectángulos.
feats <- c("Cell.size", "Bare.nuclei")
grid2 <- expand.grid(Cell.size = seq(1, 10, 0.1), Bare.nuclei = seq(1, 10, 0.1))
tp <- rpart(Class ~ Cell.size + Bare.nuclei, tr, method = "class", control = ctl(3))
tfull2 <- rpart(Class ~ Cell.size + Bare.nuclei, tr, method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
for (mt in list(list(tp, "depth-3 tree"), list(tfull2, "unpruned tree"))) { m <- mt[[1]]
  Z <- predict(m, grid2, type = "class")
  plot(grid2$Cell.size, grid2$Bare.nuclei, col = ifelse(Z == "malignant", "#fbe3d3", "#dfe7f1"), pch = 15, cex = 0.5,
       xlab = feats[1], ylab = feats[2], main = sprintf("%s: test acc %.3f (%d leaves)", mt[[2]], acc(predict(m, te, type = "class"), te$Class), nleaf(m)))
  jit <- function(v) v + runif(length(v), -0.3, 0.3)
  points(jit(te$Cell.size), jit(te$Bare.nuclei), col = ifelse(te$Class == "malignant", ORANGE, BLUE), pch = 19, cex = 0.5) }



<p class="caption"><b>Figura 12.</b> Regiones de decisión de un árbol de profundidad 3 (izquierda) y de un árbol sin podar (derecha) sobre dos características, con los casos de prueba superpuestos con un pequeño jitter. Toda frontera es horizontal o vertical. El árbol sin podar talla franjas delgadas alrededor de casos de entrenamiento individuales; el árbol de profundidad 3 dibuja unas pocas cajas grandes y es aproximadamente igual de exacto.</p>

Donde la frontera verdadera es oblicua —la malignidad aumenta con el tamaño celular y el puntaje de núcleos desnudos a la vez—, un árbol solo puede aproximarla con escalones, y cuanto más finos son los escalones, menos casos de entrenamiento sustentan cada uno. Compara con las fronteras suaves de SVM y k-NN en la Unidad 3; los bosques suavizan la escalera promediando muchas de ellas.

<div class="exercise"><h4>Ejercicio 3</h4><p>Reemplaza <code>feats</code> por dos características débiles, como <code>"Mitoses"</code> y <code>"Epith.c.size"</code>: ¿cuántas hojas hace crecer el árbol sin podar y cuál es su exactitud de prueba? Luego fija <code>minbucket = 10</code> en ambos árboles. ¿Cómo cambia la imagen y por qué esta regla de detención suele ser preferible a un límite de profundidad?</p></div>

## 7. Bosques aleatorios

### 7.1 Bagging: promediar predictores inestables

En 1996, Breiman propuso un remedio para la inestabilidad casi vergonzosamente simple: entrenar muchos árboles, cada uno sobre una <strong>muestra bootstrap</strong> distinta (\(n\) casos extraídos <em>con reemplazo</em> de los \(n\) disponibles), y combinarlos por votación o promedio. Lo llamó <strong>bagging</strong>, por <em>bootstrap aggregating</em>, y mostró que ayuda considerablemente a los predictores inestables mientras deja casi sin cambio a los estables, como k-NN.<sup class="c"><a href="#r11">[11]</a></sup> La razón es un hecho sobre los promedios: si \(B\) predictores tienen cada uno varianza \(\sigma^2\) y correlación por pares \(\rho\),

\[ \operatorname{Var}\!\left(\frac{1}{B}\sum_{b=1}^{B} \hat f_b\right) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2 . \]

El segundo término se anula a medida que \(B\) crece: promediar elimina la parte <em>independiente</em> de la variabilidad. El primero no: lo que los árboles tienen en común sobrevive.<sup class="c"><a href="#r23">[23]</a></sup> La celda simula la fórmula con \(\hat f_b = \sqrt{\rho}\,c + \sqrt{1-\rho}\,e_b\) (\(c\) compartido, \(e_b\) independientes, todos normales estándar).



In [ ]:
# Varianza del promedio de B predictores con correlación por pares rho: simulación frente a rho + (1-rho)/B.
set.seed(0)
Bs <- c(1, 2, 3, 5, 8, 12, 20, 35, 60, 100, 200); reps <- 4000
plot(NA, xlim = range(Bs), ylim = c(0, 1.05), log = "x", xlab = "number of averaged predictors B",
     ylab = "variance of the average", main = "Averaging removes independent, not shared, variability")
cols <- c(GREEN, BLUE, ORANGE, GREY); rhos <- c(0, 0.25, 0.5, 0.75)
for (j in seq_along(rhos)) { rho <- rhos[j]
  sim <- sapply(Bs, function(B) { shared <- rnorm(reps); indiv <- matrix(rnorm(reps * B), reps, B)
    var(rowMeans(sqrt(rho) * shared + sqrt(1 - rho) * indiv)) })
  points(Bs, sim, pch = 19, col = cols[j], cex = 0.7); lines(Bs, rho + (1 - rho) / Bs, col = cols[j], lwd = 1.5) }
legend("topright", bty = "n", cex = 0.8, lwd = 1.5, col = cols, title = "pairwise correlation", legend = sprintf("rho = %.2f", rhos))
cat("floor as B -> infinity equals rho; at B=200 the formula gives:",
    paste(sprintf("rho=%.2f:%.3f", rhos, rhos + (1 - rhos) / 200), collapse = "  "), "\n")



<p class="caption"><b>Figura 13.</b> Varianza del promedio de \(B\) predictores de varianza unitaria para cuatro correlaciones por pares; puntos simulados, líneas según la fórmula. Con \(\rho = 0\), la varianza cae como \(1/B\); con \(\rho = 0.5\), alcanza un piso de 0.5 tras unas pocas decenas de predictores.</p>

Los árboles crecidos sobre muestras bootstrap de los mismos datos están fuertemente correlacionados —tienden a elegir la misma característica fuerte en la raíz—, de modo que sus errores son compartidos. Para reducir \(\rho\) hay que hacer que los árboles sean <em>más distintos</em>, aun a costa de que cada uno sea individualmente peor.

### 7.2 Subconjuntos aleatorios de características

Esa es la idea que Ho introdujo en 1995 como el <em>método del subespacio aleatorio</em> (random subspace method)<sup class="c"><a href="#r13">[13]</a>,<a href="#r14">[14]</a></sup> y que Breiman combinó con el bagging en 2001 para definir el <strong>bosque aleatorio</strong>:<sup class="c"><a href="#r12">[12]</a></sup> en <em>cada nodo</em> de cada árbol se considera solo un subconjunto aleatorio de \(m\) de las \(p\) características (<code>mtry</code>; por defecto \(m = \sqrt{p}\) en clasificación, es decir, \(m = 3\) de las nueve características aquí). Un árbol al que se le prohíbe usar <em>cell shape</em> en algún nodo debe encontrar la siguiente mejor característica, y los árboles que difieren en la raíz divergen por debajo de ella. Cada árbol es algo peor; el conjunto es mejor. Los árboles se hacen crecer hasta su profundidad completa sin poda —su sobreajuste individual se compensa al promediar—, de modo que un bosque tiene esencialmente dos parámetros, \(m\) y \(B\), y es poco sensible a ambos en un rango amplio. La cota de Breiman hace explícito el compromiso: el error de generalización es a lo sumo \(\bar\rho\,(1 - s^2)/s^2\), con \(\bar\rho\) la correlación media entre árboles y \(s\) su fuerza promedio.<sup class="c"><a href="#r12">[12]</a></sup> La implementación en R es el paquete <code>randomForest</code> de Liaw y Wiener.<sup class="c"><a href="#r35">[35]</a></sup>

### 7.3 Error out-of-bag: una estimación de validación gratuita

Cada muestra bootstrap omite en promedio una fracción \((1 - 1/n)^n \approx e^{-1} \approx 0.368\) de los casos de entrenamiento: los casos <strong>out-of-bag</strong> (OOB) de ese árbol. Así, cualquier caso de entrenamiento es out-of-bag para alrededor de un tercio de los árboles, que pueden clasificarlo <em>como si fuera nuevo</em>. Comparar el voto OOB de cada caso con su etiqueta da el <strong>error OOB</strong>, una estimación del error de generalización sin conjunto de prueba ni validación cruzada, sin costo alguno; Breiman mostró que es aproximadamente tan exacta como un conjunto de prueba del tamaño del conjunto de entrenamiento.<sup class="c"><a href="#r15">[15]</a>,<a href="#r12">[12]</a></sup> La celda ajusta 300 árboles y compara la exactitud OOB, la de validación cruzada y la de prueba.



In [ ]:
# Un bosque aleatorio sobre la parte de entrenamiento: exactitud out-of-bag frente a validación cruzada frente al conjunto de prueba reservado.
cat(sprintf("fraction left out of one bootstrap sample: (1 - 1/%d)^%d = %.3f  (e^-1 = %.3f)\n", nn, nn, (1 - 1 / nn)^nn, exp(-1)))
set.seed(1); rf <- randomForest(Class ~ ., tr, ntree = 300, importance = TRUE)
set.seed(1); fold <- sample(rep(1:5, length.out = nrow(tr)))
cv_rf <- mean(sapply(1:5, function(i) { m <- randomForest(Class ~ ., tr[fold != i, ], ntree = 300)
  acc(predict(m, tr[fold == i, ]), tr$Class[fold == i]) }))
print(data.frame(estimate = c("OOB accuracy (training cases, OOB trees)", "5-fold CV accuracy (training part)",
                              "accuracy on the 205 held-out test cases", "accuracy on the training cases themselves"),
                 value = round(c(1 - rf$err.rate[300, "OOB"], cv_rf, acc(predict(rf, te), te$Class), acc(predict(rf, tr), tr$Class)), 3)), row.names = FALSE)
cat("mtry (features tried per split):", rf$mtry, "= sqrt(9)\n")



La estimación OOB, la estimación por validación cruzada y la exactitud de prueba concuerdan con una diferencia de uno o dos puntos porcentuales, mientras que la exactitud sobre los casos de entrenamiento es 1.0, como debe ser para árboles sin podar, y no dice nada. En la práctica, la estimación OOB reemplaza a la validación cruzada en los bosques: no cuesta nada, usa todos los datos y puede monitorearse a medida que se agregan árboles, lo que responde a la pregunta "¿cuántos árboles?".



In [ ]:
# Error OOB a medida que se agregan árboles: randomForest lo registra después de cada árbol en rf$err.rate.
single_err <- 1 - mean(kfold(30))                           # un solo árbol profundo, error por VC de 5 particiones
plot(1:300, rf$err.rate[, "OOB"], type = "l", col = BLUE, lwd = 1.5, log = "x", ylim = c(0, max(rf$err.rate[, "OOB"])),
     xlab = "number of trees B", ylab = "error rate", main = "Out-of-bag error versus number of trees")
abline(h = single_err, col = ORANGE, lty = 2)
legend("topright", bty = "n", cex = 0.8, col = c(BLUE, ORANGE), lwd = c(1.5, 1), lty = c(1, 2),
       legend = c("OOB error of the forest", sprintf("single tree (5-fold CV error %.3f)", single_err)))
cat(sprintf("OOB error at B = 10, 50, 100, 300: %.3f, %.3f, %.3f, %.3f\n",
    rf$err.rate[10, "OOB"], rf$err.rate[50, "OOB"], rf$err.rate[100, "OOB"], rf$err.rate[300, "OOB"]))



<p class="caption"><b>Figura 14.</b> Error out-of-bag a medida que se agregan árboles, frente al error por validación cruzada de un solo árbol (línea discontinua). El error del bosque cae abruptamente durante las primeras decenas de árboles y se estabiliza a partir de unos cien, por debajo del error del árbol individual.</p>

Con muy pocos árboles, muchos casos tienen solo uno o dos votos OOB y la estimación es ruidosa; hacia los 50 a 100 árboles ya alcanzó su meseta, y 300 a 500 es un valor por defecto seguro para este tamaño. El número de árboles no puede sobreajustarse: más árboles nunca aumentan el error esperado, solo el tiempo.

### 7.4 Importancia de las características: basada en impureza frente a permutación

Un bosque hereda la importancia por impureza de la Sección 6.8, promediada sobre los árboles, y ofrece una segunda medida: la <strong>importancia por permutación</strong>. Se permutan al azar los valores de una característica entre los casos out-of-bag —lo que destruye su relación con la etiqueta pero conserva su distribución— y se mide la caída de la exactitud. Una característica que el modelo no usa da una caída de cero cualquiera sea su número de puntos de corte, lo que elimina el primer sesgo de la Sección 6.8.<sup class="c"><a href="#r12">[12]</a>,<a href="#r16">[16]</a></sup> La correlación persiste: si se permuta <em>cell size</em> pero <em>cell shape</em> queda intacta, el bosque conserva casi la misma información y la caída es pequeña para cada miembro de un grupo correlacionado aun cuando el grupo sea esencial; las pruebas de permutación condicionales y corregidas abordan este problema.<sup class="c"><a href="#r17">[17]</a>,<a href="#r18">[18]</a></sup> Con <code>importance = TRUE</code>, <code>randomForest</code> informa ambas: <code>MeanDecreaseGini</code> (MDI) y <code>MeanDecreaseAccuracy</code> (permutación OOB).



In [ ]:
# Importancia basada en impureza (MeanDecreaseGini) frente a importancia por permutación (MeanDecreaseAccuracy). No coinciden.
imp <- importance(rf)[, c("MeanDecreaseAccuracy", "MeanDecreaseGini")]
imp <- imp[order(imp[, "MeanDecreaseGini"]), ]
par(mfrow = c(1, 2), mar = c(4, 7, 2, 1))
barplot(imp[, "MeanDecreaseGini"], horiz = TRUE, las = 1, col = BLUE, cex.names = 0.8, main = "impurity importance (MDI)", xlab = "mean decrease in Gini")
barplot(imp[, "MeanDecreaseAccuracy"], horiz = TRUE, las = 1, col = ORANGE, cex.names = 0.8, main = "permutation importance (OOB)", xlab = "mean decrease in accuracy")
cat("top-3 by MDI:        ", paste(rev(rownames(imp))[1:3], collapse = ", "), "\n")
cat("top-3 by permutation:", paste(rownames(imp)[order(-imp[, "MeanDecreaseAccuracy"])][1:3], collapse = ", "), "\n")



<p class="caption"><b>Figura 15.</b> Importancia por impureza (izquierda) e importancia por permutación (derecha) de las nueve características. Ambas coinciden en que la forma celular, el tamaño celular y los núcleos desnudos dominan, pero los ordenamientos difieren: los núcleos desnudos y el grosor del grupo celular suben con la permutación, porque contienen información que no duplica el par correlacionado de tamaño y forma.</p>

La importancia por impureza dice "estas características se usaron mucho"; la importancia por permutación dice "las predicciones dependen de esta característica <em>dado que las demás están disponibles</em>". Con mediciones casi duplicadas, la primera se reparte entre el grupo y la segunda es pequeña para cada miembro. Ninguna de las dos dice "esta característica causa la malignidad". Para una explicación por paciente, la herramienta moderna es la descomposición en valores de Shapley para árboles, que va más allá de esta unidad.<sup class="c"><a href="#r31">[31]</a></sup>

### 7.5 Dependencia parcial: ¿qué hace el bosque con una característica?

Un bosque no tiene coeficientes, pero se puede preguntar cómo cambia su predicción, <em>en promedio</em>, cuando se hace variar una característica dejando las demás tal como se observaron: la <strong>dependencia parcial</strong> respecto de la característica \(j\) en el valor \(v\):<sup class="c"><a href="#r19">[19]</a></sup>

\[ \overline{f}_j(v) = \frac{1}{n} \sum_{i=1}^{n} \hat f\big(x_{i1}, \dots, x_{i,j-1},\, v,\, x_{i,j+1}, \dots, x_{ip}\big), \]

que se calcula más abajo directamente a partir de la definición para las dos características de la Figura 12.



In [ ]:
# Dependencia parcial de la P(maligno) predicha por el bosque respecto de dos características, calculada a partir de la definición.
partial_dependence <- function(model, data, feature, grid)
  sapply(grid, function(v) { d <- data; d[[feature]] <- v; mean(predict(model, d, type = "prob")[, "malignant"]) })
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
for (fe in c("Cell.size", "Bare.nuclei")) { grid <- 1:10; pdv <- partial_dependence(rf, tr, fe, grid)
  plot(grid, pdv, type = "l", col = BLUE, lwd = 2, ylim = c(0, 1), xlab = fe, ylab = "mean predicted P(malignant)", main = paste("partial dependence on", fe))
  rug(jitter(tr[[fe]]), col = GREY)
  cat(sprintf("%s: P(malignant) rises from %.2f at %d to %.2f at %d\n", fe, pdv[1], grid[1], pdv[length(pdv)], grid[length(grid)])) }



<p class="caption"><b>Figura 16.</b> Dependencia parcial de la probabilidad de malignidad predicha por el bosque respecto de <em>cell size</em> y <em>bare nuclei</em>; las marcas indican los valores de entrenamiento. Ambas curvas suben en el rango donde los árboles ubican la mayoría de los umbrales y se aplanan fuera de él: una versión suave de las funciones escalonadas de la Sección 6.9, porque cientos de escalones en umbrales ligeramente distintos promedian una rampa.</p>

Las curvas suben abruptamente, pero no hasta 1: fijar una característica en un valor de aspecto maligno no cambia la predicción para un paciente cuyas otras características parecen benignas; y esa es también la advertencia, ya que tales combinaciones podrían no darse nunca en los datos. Aun así, una curva de dependencia parcial muestra la dirección, la forma y el rango en que una característica importa, algo que ningún puntaje de importancia muestra.

### 7.6 El bosque frente al árbol individual y los métodos de la Unidad 3

El último experimento compara, bajo el protocolo de S1, el árbol podado de la Sección 6.7, el bosque y tres clasificadores de la Unidad 3 —Bayes ingenuo, k vecinos más cercanos y la SVM con kernel RBF— mediante validación cruzada de 5 particiones sobre la parte de entrenamiento (exactitud y ROC AUC) y una vez sobre la parte de prueba.



In [ ]:
# Comparación bajo el protocolo de S1: VC de 5 particiones sobre la parte de entrenamiento (exactitud, ROC AUC) y luego una evaluación en el conjunto de prueba.
auc_te <- function(prob_mal) as.numeric(auc(roc(te$Class, prob_mal, quiet = TRUE, levels = c("benign", "malignant"), direction = "<")))
pos <- function(m, d) {                                     # P(maligno) según cada modelo
  if (inherits(m, "randomForest")) predict(m, d, type = "prob")[, "malignant"]
  else if (inherits(m, "naiveBayes")) predict(m, d, type = "raw")[, "malignant"]
  else if (inherits(m, "rpart")) predict(m, d)[, "malignant"] else attr(predict(m, d, probability = TRUE), "probabilities")[, "malignant"]
}
set.seed(1); fit_list <- list(
  "decision tree, pruned (Sec. 6.7)" = prune(tcp, cp = cp_opt * 1.0001),
  "random forest, 300 trees"         = randomForest(Class ~ ., tr, ntree = 300),
  "naive Bayes (Unit 3)"             = naiveBayes(Class ~ ., tr),
  "SVM, RBF kernel (Unit 3)"         = svm(Class ~ ., tr, kernel = "radial", probability = TRUE))
cv5 <- function(fitfun) { set.seed(1); fo <- sample(rep(1:5, length.out = nrow(tr)))
  mean(sapply(1:5, function(i) { m <- fitfun(tr[fo != i, ]); acc(predict(m, tr[fo == i, ], type = if (inherits(m, "rpart")) "class" else "class"), tr$Class[fo == i]) })) }
cvfuns <- list(function(d) prune(rpart(Class ~ ., d, method = "class", control = rpart.control(cp = 0, xval = 0)), cp = cp_opt * 1.0001),
               function(d) randomForest(Class ~ ., d, ntree = 300), function(d) naiveBayes(Class ~ ., d),
               function(d) svm(Class ~ ., d, kernel = "radial"))
# k-NN se trata por separado (requiere escalamiento)
sc <- scale(tr[, 1:9]); scte <- scale(te[, 1:9], center = attr(sc, "scaled:center"), scale = attr(sc, "scaled:scale"))
comp <- do.call(rbind, Map(function(nm, m, cf) data.frame(model = nm, CV_accuracy = round(cv5(cf), 3),
                 CV_ROC_AUC = round(auc_te(pos(m, te)), 3), test_accuracy = round(acc(predict(m, te, type = if (inherits(m, "rpart")) "class" else "class"), te$Class), 3)),
                 names(fit_list), fit_list, cvfuns))
knn5 <- knn(sc, scte, tr$Class, k = 5)
comp <- rbind(comp, data.frame(model = "k-NN, k=5, standardised (Unit 3)", CV_accuracy = NA, CV_ROC_AUC = NA, test_accuracy = round(acc(knn5, te$Class), 3)))
print(comp, row.names = FALSE)



El bosque mejora al árbol podado en un par de puntos de exactitud por validación cruzada y en varios puntos de ROC AUC: promediar árboles inestables rinde frutos. Entre los métodos fuertes, las diferencias quedan a menos de una desviación estándar entre sí en estos 478 casos de entrenamiento; el árbol individual queda claramente último. El panorama es representativo: en una comparación de 179 clasificadores sobre 121 conjuntos de datos, los bosques aleatorios fueron la familia que con mayor frecuencia resultó la mejor o cercana a la mejor, con las SVM muy cerca,<sup class="c"><a href="#r22">[22]</a></sup> y evaluaciones comparativas recientes sobre datos tabulares de tamaño mediano todavía encuentran a los conjuntos de árboles por delante de las redes neuronales profundas.<sup class="c"><a href="#r21">[21]</a>,<a href="#r32">[32]</a></sup> Lo que el bosque ofrece y la SVM no es la ausencia de escalamiento, los tipos de características mixtos, una estimación del error gratuita y herramientas de importancia y de dependencia parcial; lo que cede frente al árbol individual es la lista legible de reglas: el compromiso que Breiman describió en su ensayo sobre las "dos culturas" del modelado estadístico.<sup class="c"><a href="#r30">[30]</a></sup>



In [ ]:
# La superficie de probabilidad del bosque sobre las dos características de la Figura 12, junto a la del árbol sin podar.
set.seed(1); rf2d <- randomForest(Class ~ Cell.size + Bare.nuclei, tr, ntree = 300)
gx <- seq(1, 10, 0.15); gg <- expand.grid(Cell.size = gx, Bare.nuclei = gx)
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
for (mt in list(list(tfull2, "unpruned tree"), list(rf2d, "random forest (300 trees)"))) { m <- mt[[1]]
  P <- matrix(predict(m, gg, type = "prob")[, "malignant"], length(gx))
  image(gx, gx, P, col = hcl.colors(20, "Blue-Red"), xlab = feats[1], ylab = feats[2], zlim = c(0, 1),
        main = sprintf("%s, test acc %.3f", mt[[2]], acc(predict(m, te, type = if (inherits(m, "rpart")) "class" else "class"), te$Class)))
  contour(gx, gx, P, levels = 0.5, add = TRUE, drawlabels = FALSE, lwd = 1.5)
  jit <- function(v) v + runif(length(v), -0.3, 0.3)
  points(jit(te$Cell.size), jit(te$Bare.nuclei), col = ifelse(te$Class == "malignant", ORANGE, BLUE), pch = 19, cex = 0.5) }



<p class="caption"><b>Figura 17.</b> Probabilidad de malignidad predicha sobre el plano de dos características, para el árbol sin podar (izquierda) y el bosque (derecha); la línea negra es el contorno de 0.5. Las probabilidades del árbol son casi 0 o 1 en todas partes, con islas rectangulares; las del bosque varían de forma gradual y las islas desaparecen al promediar.</p>

Promediar unos cientos de escaleras produce una frontera más suave y una <em>incertidumbre graduada</em>: la región pálida cerca de la frontera es donde los árboles discrepan, y un clínico debería leer una probabilidad de 0.55 de forma distinta a una de 0.98.

<div class="box key"><h4>Los bosques aleatorios en un párrafo</h4><p>Haz crecer \(B\) árboles sin podar, cada uno sobre una muestra bootstrap, considerando en cada nodo solo \(m \approx \sqrt{p}\) características aleatorias; predice por votación o promedio. El bootstrap y el submuestreo de características decorrelacionan los árboles para que el promedio reduzca la varianza (Figura 13); los casos out-of-bag estiman el error gratuitamente (Sección 7.3); basta con que \(B\) alcance la meseta (Figura 14); el precio es la pérdida de un árbol individual legible, compensada en parte por las medidas de importancia y la dependencia parcial.</p></div>

## 8. Más allá del programa: gradient boosting

<div class="box bridge"><h4>Fuera del programa oficial</h4><p>Esta sección va un paso más allá de la unidad porque el método es, en la práctica actual, la opción por defecto para datos tabulares, y porque es la contraparte natural del bosque: ambos son conjuntos de árboles que combinan sus árboles de maneras opuestas.</p></div>

Un bosque hace crecer sus árboles de forma <em>independiente</em> y los promedia para reducir la varianza. El <strong>gradient boosting</strong>, formulado por Friedman en 2001, hace crecer árboles poco profundos de forma <em>secuencial</em>, cada uno ajustado al gradiente negativo de la pérdida del conjunto acumulado hasta ese momento, y los suma con una tasa de aprendizaje pequeña, reduciendo el <em>sesgo</em> paso a paso.<sup class="c"><a href="#r19">[19]</a></sup> XGBoost (2016) hizo el método rápido y regularizado, y él y sus sucesores han ganado desde entonces la mayoría de las competencias sobre datos tabulares.<sup class="c"><a href="#r20">[20]</a></sup> Las comparaciones sistemáticas encuentran que los árboles con boosting y los bosques siguen por delante de las redes profundas en tablas de tamaño mediano, al parecer porque los conjuntos de árboles ignoran las características no informativas y representan funciones irregulares que las redes suavizan.<sup class="c"><a href="#r21">[21]</a>,<a href="#r32">[32]</a></sup> A diferencia de un bosque, el boosting <em>sí puede</em> sobreajustarse a medida que se agregan árboles, por lo que su número debe elegirse por validación o con early stopping. R tiene paquetes dedicados (<code>gbm</code>, <code>xgboost</code>), pero para ceñirse a la pequeña caja de herramientas del curso, la celda implementa el algoritmo desde cero en unas quince líneas: un gradient boosting logístico cuyos aprendices base son árboles de regresión <code>rpart</code> de profundidad 2 ajustados a los residuos \(y - p\).



In [ ]:
# Gradient boosting desde cero: árboles de regresión rpart de profundidad 2 ajustados secuencialmente al gradiente logístico.
gb_fit <- function(train, M = 200, lr = 0.05, depth = 2) {
  y <- as.numeric(train$Class == "malignant"); X <- train[, 1:9]
  F0 <- log(mean(y) / (1 - mean(y))); Fv <- rep(F0, nrow(train)); trees <- vector("list", M)
  for (m in 1:M) { p <- 1 / (1 + exp(-Fv)); g <- y - p                       # gradiente negativo de la pérdida logarítmica (log-loss)
    t <- rpart(g ~ ., cbind(X, g = g), method = "anova", control = rpart.control(maxdepth = depth, cp = 0, xval = 0))
    Fv <- Fv + lr * predict(t); trees[[m]] <- t }
  list(F0 = F0, trees = trees, lr = lr)
}
gb_prob <- function(mod, newX) { Fv <- rep(mod$F0, nrow(newX)); for (t in mod$trees) Fv <- Fv + mod$lr * predict(t, newX); 1 / (1 + exp(-Fv)) }
set.seed(1); gb <- gb_fit(tr, M = 200, lr = 0.05, depth = 2)
pte <- gb_prob(gb, te[, 1:9])
cat(sprintf("gradient boosting (200 depth-2 trees): train acc %.3f, test acc %.3f, test ROC AUC %.3f\n",
    mean((gb_prob(gb, tr[, 1:9]) > 0.5) == (tr$Class == "malignant")), mean((pte > 0.5) == (te$Class == "malignant")), auc_te(pte)))
cat(sprintf("for comparison, the random forest of Section 7.3: test ROC AUC %.3f\n", auc_te(predict(rf, te, type = "prob")[, "malignant"])))



El boosting queda en el mismo grupo que el bosque y la SVM, dentro del ruido entre particiones en estos 478 casos de entrenamiento: un conjunto de árboles poco profundos escrito a mano alcanza un ROC AUC de prueba cercano al del bosque. En tablas más grandes y desordenadas —muchas características no informativas, valores faltantes, tipos mixtos— la distancia entre los conjuntos de árboles y los métodos de la Unidad 3 suele ampliarse a favor de los conjuntos; en problemas pequeños y limpios, una SVM bien ajustada suele ser igual de buena y más fácil de defender. La comparación debe ejecutarse bajo el protocolo de S1, no darse por supuesta.

## 9. Aplicaciones en medicina y bioingeniería

Los árboles de decisión entraron en la medicina antes que en el aprendizaje automático, porque el protocolo de un clínico <em>es</em> un árbol.

<strong>Triaje del dolor torácico (1982).</strong> Goldman y colaboradores aplicaron partición recursiva —CART, con Olshen entre los autores— a 482 pacientes de urgencias para derivar un protocolo que predecía el infarto de miocardio a partir del electrocardiograma y la anamnesis; probado prospectivamente en otros 468 pacientes, el protocolo combinado con el juicio de los médicos mantuvo la sensibilidad y elevó la especificidad del ingreso a cuidados intensivos del 67 % al 77 %.<sup class="c"><a href="#r27">[27]</a></sup> La monografía de CART, publicada dos años después, comienza con ejemplos médicos del mismo período, incluido el pronóstico de pacientes con infarto.<sup class="c"><a href="#r7">[7]</a></sup> El protocolo podía imprimirse en una tarjeta y usarse junto a la cama del paciente.

<strong>Reglas de decisión clínica (1992).</strong> Las reglas de tobillo de Ottawa, derivadas por Stiell y colaboradores a partir de 750 pacientes, son una secuencia de preguntas de sí/no sobre la sensibilidad ósea a la palpación y la capacidad de apoyar el peso, que identifica a los pacientes que no necesitan radiografía: en términos operativos, un árbol de decisión poco profundo ejecutado a mano; en el estudio de derivación tuvieron 100 % de sensibilidad para fractura.<sup class="c"><a href="#r26">[26]</a></sup> La mayoría de las reglas de decisión en medicina de urgencias tienen esta forma porque un árbol puede memorizarse y aplicarse bajo presión de tiempo: la interpretabilidad de la Sección 6.10 es lo que las hace utilizables.

<strong>Genómica (2006).</strong> Con miles de características y decenas de pacientes —microarreglos de expresión génica—, los árboles individuales no tienen ninguna posibilidad y la mayoría de los métodos de la Unidad 3 requieren una fuerte selección de características. Díaz-Uriarte y Alvarez de Andrés mostraron que los bosques aleatorios clasifican tales datos tan bien como las mejores alternativas sin preselección, y propusieron seleccionar genes descartando iterativamente los genes menos importantes mientras se monitorea el error OOB.<sup class="c"><a href="#r25">[25]</a></sup> Los sesgos de las medidas de importancia (Secciones 6.8 y 7.4) se documentaron precisamente a raíz de su uso para jerarquizar genes.<sup class="c"><a href="#r16">[16]</a>,<a href="#r18">[18]</a></sup>

<div class="box warn"><h4>Cómo leer un artículo basado en árboles</h4><p>¿Se podó el árbol, y cómo se eligió la profundidad o \(\alpha\): con los datos de prueba? ¿La exactitud informada es OOB, por validación cruzada o en un conjunto independiente? Si se interpretan importancias, ¿están correlacionadas las características y se usó la importancia por permutación? ¿Son plausibles las reglas para un clínico, o sugieren shortcut learning (Unidad 1)?</p></div>

## 10. Resumen

Ahora deberías poder explicar, sin apuntes:

- La definición recursiva de un árbol binario y su vocabulario (raíz, hoja, profundidad, altura en aristas con −1 para el árbol vacío; lleno, perfecto, completo).
- Las tres representaciones; que <code>rpart</code> almacena un árbol como un data frame cuyos números de nodo obedecen la regla \(2i\) / \(2i+1\).
- Los cuatro recorridos; que el preorden imprime un árbol como reglas y que el inorden entrega ordenadas las claves de un árbol de búsqueda.
- Que un árbol perfecto de altura \(h\) tiene \(2^{h+1}-1\) nodos, de modo que la altura de un árbol de \(n\) nodos está entre \(\log_2(n+1)-1\) y \(n-1\).
- El invariante BST; inserción y búsqueda en \(O(h)\); eliminación mediante el sucesor inorden; la inserción ordenada da \(h = n-1\) y la inserción aleatoria \(h \propto \log n\).
- Qué hace una rotación; qué garantizan los árboles AVL, rojo–negro y B, y dónde se usan.
- Gini, entropía y error de clasificación, y por qué se usan las cóncavas para hacer crecer el árbol; el algoritmo voraz CART y el barrido de umbrales; por qué los árboles no necesitan escalamiento.
- Profundidad frente a sobreajuste; poda por costo-complejidad; los dos sesgos de la importancia por impureza; los árboles de regresión como funciones constantes por tramos; inestabilidad y fronteras alineadas con los ejes.
- El bagging y \(\rho\sigma^2 + (1-\rho)\sigma^2/B\); los subconjuntos aleatorios de características como forma de reducir \(\rho\); la estimación OOB y \(e^{-1}\); importancia por impureza frente a importancia por permutación; dependencia parcial; cómo se compara el bosque con un árbol individual y con la Unidad 3.

## 11. Autoevaluación

<details><summary>1. Un árbol binario tiene 20 nodos. ¿Cuáles son las alturas mínima y máxima posibles, y qué formas las alcanzan?</summary>Mínima \(\lceil \log_2 21 \rceil - 1 = 4\) (un árbol completo: 15 nodos en los cuatro primeros niveles, 5 en el quinto). Máxima 19 (una cadena).</details>

<details><summary>2. Da los recorridos en preorden, inorden y postorden del árbol con raíz 8, hijo izquierdo 3 (con hijos 1 y 6) e hijo derecho 10 (solo con hijo derecho 14). ¿Es un BST?</summary>Preorden 8 3 1 6 10 14; inorden 1 3 6 8 10 14; postorden 1 6 3 14 10 8. Sí: la secuencia inorden es estrictamente creciente.</details>

<details><summary>3. En la representación en vector (contando desde 0), un nodo está almacenado en el índice 12. ¿Dónde están su padre y sus hijos?</summary>Padre en \(\lfloor 11/2 \rfloor = 5\); hijos en 25 y 26. (En la numeración de <code>rpart</code>, que cuenta desde uno, los hijos del nodo \(i\) son \(2i\) y \(2i+1\).)</details>

<details><summary>4. ¿Por qué insertar registros de pacientes en un BST por fecha de ingreso creciente produce una estructura lenta, y cuáles son dos soluciones?</summary>Las claves crecientes siempre van a la derecha, lo que da una cadena de altura \(n-1\) y búsqueda en \(O(n)\). Soluciones: un árbol autobalanceado (AVL, rojo–negro, árbol B), o la inserción con la mediana primero si todas las claves se conocen de antemano.</details>

<details><summary>5. Se aplica una rotación a la derecha en y con hijo izquierdo x. ¿Qué subárbol cambia de padre y por qué la secuencia inorden no cambia?</summary>El subárbol derecho de x (claves entre x e y) pasa a ser el subárbol izquierdo de y; el inorden antes y después es (izquierda de x), x, (ese subárbol), y, (derecha de y).</details>

<details><summary>6. Un nodo tiene 8 casos malignos y 2 benignos. Calcula Gini y entropía. Una división envía los 2 benignos y 2 malignos a la izquierda y 6 malignos a la derecha: calcula la disminución de Gini.</summary>Gini \(= 1 - (0.64 + 0.04) = 0.32\); entropía \(\approx 0.722\) bits. Izquierda: Gini 0.5; derecha: 0. Ponderado \(0.4 \times 0.5 = 0.2\); disminución \(0.12\).</details>

<details><summary>7. ¿Por qué CART y C4.5 hacen crecer los árboles con Gini o entropía en lugar de con el error de clasificación que finalmente se informa?</summary>Gini y entropía son estrictamente cóncavas, de modo que cualquier división que deje a los hijos más puros las hace disminuir aun sin cambiar una clase mayoritaria; el error de clasificación, lineal por tramos, puede quedar sin cambio ante una división útil y detendría el crecimiento prematuramente.</details>

<details><summary>8. En la Sección 6.4, el decision stump sobre el tamaño celular y el barrido exhaustivo coincidieron, pero el decision stump con las nueve características eligió la forma celular. ¿Qué dice esto sobre la estabilidad de la división en la raíz?</summary>La elección entre características casi duplicadas se decide por una pequeña diferencia en el Gini ponderado y, por tanto, por unos pocos casos de entrenamiento. Una muestra bootstrap puede invertirla, y como todas las divisiones inferiores dependen de la raíz, puede cambiar todo el árbol: la inestabilidad que aprovecha el bagging.</details>

<details><summary>9. Lectura de código: un colega selecciona la profundidad del árbol a partir de la curva naranja de exactitud de prueba de la Figura 9 en lugar de la azul. ¿Qué está mal?</summary>El conjunto de prueba se ha usado para la selección del modelo, por lo que su exactitud en la profundidad elegida tiene un sesgo optimista. Elige la profundidad por validación cruzada (u OOB) sobre la parte de entrenamiento y evalúa el conjunto de prueba una sola vez, al final.</details>

<details><summary>10. Explica en una oración cada uno qué hacen el bagging y los subconjuntos aleatorios de características con \(\rho\sigma^2 + (1-\rho)\sigma^2/B\).</summary>El bagging aumenta \(B\), lo que achica el segundo término; los subconjuntos aleatorios de características disminuyen \(\rho\) al hacer que los árboles elijan divisiones distintas, lo que baja el piso, a costa de un \(\sigma^2\) algo mayor por árbol.</details>

<details><summary>11. Con 478 casos de entrenamiento y 300 árboles, ¿aproximadamente cuántos árboles tienen un caso dado como out-of-bag, y por qué la estimación OOB no necesita un conjunto de prueba aparte?</summary>Alrededor de \(0.368 \times 300 \approx 110\). Esos árboles nunca vieron el caso, así que su voto es una predicción sobre datos no vistos; agregados sobre todos los casos, los votos estiman el error de generalización.</details>

<details><summary>12. En la Figura 15, la forma celular tiene la mayor importancia por impureza y, sin embargo, permutarla le cuesta poco al bosque. Explica el mecanismo y qué le dirías a un clínico que pregunta qué medición importa más.</summary>La importancia por impureza le da crédito a la característica elegida en muchas divisiones; la permutación mide la pérdida cuando se desordena mientras el tamaño celular, correlacionado con ella y portador de la misma información, sigue disponible. Informa las mediciones correlacionadas de tamaño y forma como grupo, di que el orden dentro de él es arbitrario y muestra curvas de dependencia parcial en lugar de un solo número.</details>

## Glosario

<dl class="gloss">
<dt>Árbol AVL (AVL tree)</dt><dd>BST autobalanceado en el que, en cada nodo, las alturas de los subárboles difieren a lo sumo en uno; altura a lo sumo de aproximadamente \(1.44\log_2 n\).</dd>
<dt>Bagging</dt><dd>Bootstrap aggregating: entrenar predictores sobre muestras bootstrap y combinarlos por votación o promedio para reducir la varianza.</dd>
<dt>Árbol binario de búsqueda (binary search tree, BST)</dt><dd>Árbol binario de claves en el que, en cada nodo, todas las claves de la izquierda son menores y todas las de la derecha mayores que la clave del nodo.</dd>
<dt>Árbol binario (binary tree)</dt><dd>Árbol con raíz con a lo sumo dos hijos por nodo, distinguidos como izquierdo y derecho; recursivamente, vacío o una raíz con dos subárboles binarios.</dd>
<dt>Muestra bootstrap (bootstrap sample)</dt><dd>\(n\) casos extraídos con reemplazo de \(n\); omite alrededor del 36.8 % de los casos distintos.</dd>
<dt>Árbol B (B-tree)</dt><dd>Árbol de búsqueda balanceado con muchas claves por nodo, diseñado para disco; la base de los índices de bases de datos.</dd>
<dt>CART (CART)</dt><dd>Árboles de clasificación y regresión (Classification and Regression Trees): partición recursiva voraz con impureza de Gini y poda por costo-complejidad; el algoritmo del paquete <code>rpart</code>.</dd>
<dt>Poda por costo-complejidad (cost-complexity pruning)</dt><dd>Elección del subárbol que minimiza el error de entrenamiento más \(\alpha\) (en <code>rpart</code>, <code>cp</code>) veces el número de hojas, con \(\alpha\) seleccionado por validación cruzada.</dd>
<dt>Árbol de decisión (decision tree)</dt><dd>Árbol binario cuyos nodos internos comparan una característica con un umbral y cuyas hojas contienen una clase o un valor; sus regiones de decisión son cajas alineadas con los ejes.</dd>
<dt>Profundidad / altura (depth / height)</dt><dd>Profundidad de un nodo: aristas desde la raíz. Altura: aristas del camino descendente más largo hasta una hoja; la altura del árbol es la de su raíz (−1 si está vacío).</dd>
<dt>Entropía / impureza de Gini (entropy / Gini impurity)</dt><dd>\(-\sum_k p_k\log_2 p_k\) (ID3/C4.5) y \(1 - \sum_k p_k^2\) (CART): las medidas de impureza cuya disminución selecciona una división.</dd>
<dt>Gradient boosting</dt><dd>Conjunto que agrega árboles poco profundos de forma secuencial, cada uno ajustado al gradiente de la pérdida acumulada hasta ese momento; reduce el sesgo.</dd>
<dt>Importancia basada en impureza, MDI (impurity-based importance)</dt><dd>Disminución total de impureza atribuible a una característica; sesgada hacia las características con muchos puntos de corte y arbitraria entre características correlacionadas.</dd>
<dt>Out-of-bag error (OOB)</dt><dd>Error de un bosque en cada caso de entrenamiento según lo predicen los árboles cuya muestra bootstrap lo excluyó.</dd>
<dt>Dependencia parcial (partial dependence)</dt><dd>Predicción promedio en función de una característica, con las demás mantenidas en sus valores observados.</dd>
<dt>Importancia por permutación (permutation importance)</dt><dd>Disminución del desempeño cuando los valores de una característica se permutan al azar.</dd>
<dt>Bosque aleatorio (random forest)</dt><dd>Árboles sin podar combinados por bagging, en los que cada división considera un subconjunto aleatorio de las características.</dd>
<dt>Árbol rojo–negro (red–black tree)</dt><dd>BST autobalanceado con un bit de color por nodo que acota la altura por \(2\log_2(n+1)\).</dd>
<dt>Rotación (rotation)</dt><dd>Reorganización local de un nodo y uno de sus hijos que cambia en uno las alturas de los subárboles preservando la secuencia inorden.</dd>
<dt>Recorrido (traversal)</dt><dd>Visitar cada nodo una vez: preorden (la raíz primero), inorden (la raíz entre los subárboles), postorden (la raíz al final), por niveles (por profundidad).</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">Knuth DE. <em>The Art of Computer Programming, Volume 1: Fundamental Algorithms</em>. 3rd ed. Addison-Wesley; 1997. Sección 2.3, Trees. Sin DOI; página de la editorial: <a href="https://www.informit.com/store/art-of-computer-programming-volume-1-fundamental-algorithms-9780201896831">informit.com</a></li>
<li id="r2">Knuth DE. <em>The Art of Computer Programming, Volume 3: Sorting and Searching</em>. 2nd ed. Addison-Wesley; 1998. Sección 6.2, Searching by comparison of keys (árboles binarios de búsqueda, árboles balanceados). Sin DOI; página de la editorial: <a href="https://www.informit.com/store/art-of-computer-programming-volume-3-sorting-and-searching-9780201896855">informit.com</a></li>
<li id="r3">Adelson-Velskii GM, Landis EM. An algorithm for the organization of information. <em>Doklady Akademii Nauk SSSR</em>. 1962;146(2):263–266 (en ruso; traducción al inglés en <em>Soviet Mathematics Doklady</em>. 1962;3:1259–1263). Sin DOI; original en Math-Net.Ru: <a href="https://www.mathnet.ru/eng/dan26964">mathnet.ru/eng/dan26964</a></li>
<li id="r4">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r5">Guibas LJ, Sedgewick R. A dichromatic framework for balanced trees. In: <em>Proceedings of the 19th Annual Symposium on Foundations of Computer Science (SFCS 1978)</em>. IEEE; 1978:8–21. <a href="https://doi.org/10.1109/SFCS.1978.3">doi:10.1109/SFCS.1978.3</a></li>
<li id="r6">Bayer R, McCreight E. Organization and maintenance of large ordered indexes. <em>Acta Informatica</em>. 1972;1(3):173–189. <a href="https://doi.org/10.1007/BF00288683">doi:10.1007/BF00288683</a></li>
<li id="r7">Breiman L, Friedman JH, Olshen RA, Stone CJ. <em>Classification and Regression Trees</em>. Wadsworth; 1984. Reeditado por Chapman and Hall/CRC; 2017. <a href="https://doi.org/10.1201/9781315139470">doi:10.1201/9781315139470</a></li>
<li id="r8">Quinlan JR. Induction of decision trees. <em>Machine Learning</em>. 1986;1(1):81–106. <a href="https://doi.org/10.1007/BF00116251">doi:10.1007/BF00116251</a></li>
<li id="r9">Quinlan JR. <em>C4.5: Programs for Machine Learning</em>. Morgan Kaufmann; 1993. El libro no tiene DOI; véase la reseña de Salzberg SL. <em>Machine Learning</em>. 1994;16(3):235–240. <a href="https://doi.org/10.1007/BF00993309">doi:10.1007/BF00993309</a></li>
<li id="r10">Hyafil L, Rivest RL. Constructing optimal binary decision trees is NP-complete. <em>Information Processing Letters</em>. 1976;5(1):15–17. <a href="https://doi.org/10.1016/0020-0190(76)90095-8">doi:10.1016/0020-0190(76)90095-8</a></li>
<li id="r11">Breiman L. Bagging predictors. <em>Machine Learning</em>. 1996;24(2):123–140. <a href="https://doi.org/10.1007/BF00058655">doi:10.1007/BF00058655</a></li>
<li id="r12">Breiman L. Random forests. <em>Machine Learning</em>. 2001;45(1):5–32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r13">Ho TK. Random decision forests. In: <em>Proceedings of the 3rd International Conference on Document Analysis and Recognition</em>. IEEE; 1995;1:278–282. <a href="https://doi.org/10.1109/ICDAR.1995.598994">doi:10.1109/ICDAR.1995.598994</a></li>
<li id="r14">Ho TK. The random subspace method for constructing decision forests. <em>IEEE Transactions on Pattern Analysis and Machine Intelligence</em>. 1998;20(8):832–844. <a href="https://doi.org/10.1109/34.709601">doi:10.1109/34.709601</a></li>
<li id="r15">Breiman L. <em>Out-of-bag estimation</em>. Informe técnico, Statistics Department, University of California, Berkeley; 1996. Sin DOI; <a href="https://www.stat.berkeley.edu/~breiman/OOBestimation.pdf">stat.berkeley.edu/~breiman/OOBestimation.pdf</a></li>
<li id="r16">Strobl C, Boulesteix A-L, Zeileis A, Hothorn T. Bias in random forest variable importance measures: illustrations, sources and a solution. <em>BMC Bioinformatics</em>. 2007;8:25. <a href="https://doi.org/10.1186/1471-2105-8-25">doi:10.1186/1471-2105-8-25</a></li>
<li id="r17">Strobl C, Boulesteix A-L, Kneib T, Augustin T, Zeileis A. Conditional variable importance for random forests. <em>BMC Bioinformatics</em>. 2008;9:307. <a href="https://doi.org/10.1186/1471-2105-9-307">doi:10.1186/1471-2105-9-307</a></li>
<li id="r18">Altmann A, Toloşi L, Sander O, Lengauer T. Permutation importance: a corrected feature importance measure. <em>Bioinformatics</em>. 2010;26(10):1340–1347. <a href="https://doi.org/10.1093/bioinformatics/btq134">doi:10.1093/bioinformatics/btq134</a></li>
<li id="r19">Friedman JH. Greedy function approximation: a gradient boosting machine. <em>Annals of Statistics</em>. 2001;29(5):1189–1232. <a href="https://doi.org/10.1214/aos/1013203451">doi:10.1214/aos/1013203451</a></li>
<li id="r20">Chen T, Guestrin C. XGBoost: a scalable tree boosting system. In: <em>Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining</em>. ACM; 2016:785–794. <a href="https://doi.org/10.1145/2939672.2939785">doi:10.1145/2939672.2939785</a></li>
<li id="r21">Grinsztajn L, Oyallon E, Varoquaux G. Why do tree-based models still outperform deep learning on typical tabular data? <em>Advances in Neural Information Processing Systems</em> 35 (NeurIPS 2022, Datasets and Benchmarks track). 2022:507–520. arXiv:2207.08815. <a href="https://doi.org/10.48550/arXiv.2207.08815">doi:10.48550/arXiv.2207.08815</a></li>
<li id="r22">Fernández-Delgado M, Cernadas E, Barro S, Amorim D. Do we need hundreds of classifiers to solve real world classification problems? <em>Journal of Machine Learning Research</em>. 2014;15(90):3133–3181. Sin DOI; <a href="https://jmlr.org/papers/v15/delgado14a.html">jmlr.org/papers/v15/delgado14a.html</a></li>
<li id="r23">Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning: Data Mining, Inference, and Prediction</em>. 2nd ed. Springer; 2009. Capítulos 9.2 (árboles), 10 (boosting) y 15 (bosques aleatorios). <a href="https://doi.org/10.1007/978-0-387-84858-7">doi:10.1007/978-0-387-84858-7</a></li>
<li id="r24">Louppe G. <em>Understanding Random Forests: From Theory to Practice</em>. Tesis doctoral, University of Liège; 2014. arXiv:1407.7502. <a href="https://doi.org/10.48550/arXiv.1407.7502">doi:10.48550/arXiv.1407.7502</a></li>
<li id="r25">Díaz-Uriarte R, Alvarez de Andrés S. Gene selection and classification of microarray data using random forest. <em>BMC Bioinformatics</em>. 2006;7:3. <a href="https://doi.org/10.1186/1471-2105-7-3">doi:10.1186/1471-2105-7-3</a></li>
<li id="r26">Stiell IG, Greenberg GH, McKnight RD, Nair RC, McDowell I, Worthington JR. A study to develop clinical decision rules for the use of radiography in acute ankle injuries. <em>Annals of Emergency Medicine</em>. 1992;21(4):384–390. <a href="https://doi.org/10.1016/S0196-0644(05)82656-3">doi:10.1016/S0196-0644(05)82656-3</a></li>
<li id="r27">Goldman L, Weinberg M, Weisberg M, Olshen R, Cook EF, Sargent RK, et al. A computer-derived protocol to aid in the diagnosis of emergency room patients with acute chest pain. <em>New England Journal of Medicine</em>. 1982;307(10):588–596. <a href="https://doi.org/10.1056/NEJM198209023071004">doi:10.1056/NEJM198209023071004</a></li>
<li id="r28">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proceedings of SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r29">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Annals of Statistics</em>. 2004;32(2):407–499 (fuente del conjunto de datos de diabetes usado en la edición en Python). <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r30">Breiman L. Statistical modeling: the two cultures (with comments and a rejoinder by the author). <em>Statistical Science</em>. 2001;16(3):199–231. <a href="https://doi.org/10.1214/ss/1009213726">doi:10.1214/ss/1009213726</a></li>
<li id="r31">Lundberg SM, Erion G, Chen H, DeGrave A, Prutkin JM, Nair B, et al. From local explanations to global understanding with explainable AI for trees. <em>Nature Machine Intelligence</em>. 2020;2:56–67. <a href="https://doi.org/10.1038/s42256-019-0138-9">doi:10.1038/s42256-019-0138-9</a></li>
<li id="r32">Shwartz-Ziv R, Armon A. Tabular data: deep learning is not all you need. <em>Information Fusion</em>. 2022;81:84–90. <a href="https://doi.org/10.1016/j.inffus.2021.11.011">doi:10.1016/j.inffus.2021.11.011</a></li>
<li id="r33">Harrison D, Rubinfeld DL. Hedonic housing prices and the demand for clean air. <em>Journal of Environmental Economics and Management</em>. 1978;5(1):81–102 (fuente de los datos de viviendas de Boston). <a href="https://doi.org/10.1016/0095-0696(78)90006-2">doi:10.1016/0095-0696(78)90006-2</a></li>
<li id="r34">Therneau TM, Atkinson EJ. <em>An Introduction to Recursive Partitioning Using the RPART Routines</em>. Informe técnico, Mayo Foundation; viñeta del paquete rpart. Sin DOI; <a href="https://cran.r-project.org/package=rpart">cran.r-project.org/package=rpart</a></li>
<li id="r35">Liaw A, Wiener M. Classification and regression by randomForest. <em>R News</em>. 2002;2(3):18–22. Sin DOI; <a href="https://cran.r-project.org/doc/Rnews/">cran.r-project.org/doc/Rnews</a></li>
<li id="r36">James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. Capítulo 8, métodos basados en árboles. <a href="https://doi.org/10.1007/978-1-0716-1418-1">doi:10.1007/978-1-0716-1418-1</a> (PDF gratuito en statlearning.com)</li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>Cormen TH, Leiserson CE, Rivest RL, Stein C. <em>Introduction to Algorithms</em>. 4th ed. MIT Press; 2022. — Capítulos 12 (árboles binarios de búsqueda) y 13 (árboles rojo–negro), para un tratamiento completo, con eliminación y rotaciones, de las Secciones 2–4.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Capítulo 9.2 para CART, Capítulo 10 para boosting, Capítulo 15 para bosques aleatorios; la fuente de la fórmula de varianza de la Sección 7.1.<sup class="c"><a href="#r23">[23]</a></sup></li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — Capítulo 8, métodos basados en árboles: el tratamiento más accesible de las Secciones 6–8, en R.<sup class="c"><a href="#r36">[36]</a></sup></li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. — la referencia para <code>rpart</code>, <code>randomForest</code> y las demás funciones de modelado de R usadas aquí.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>Therneau TM, Atkinson EJ. <em>An Introduction to Recursive Partitioning Using the RPART Routines</em> — la viñeta de <code>rpart</code>, con la matemática de CART y la tabla de complejidad de la Sección 6.7.<sup class="c"><a href="#r34">[34]</a></sup></li>
<li>Documentación del paquete <code>randomForest</code> y páginas de Random Forests de Breiman &amp; Cutler — <a href="https://www.stat.berkeley.edu/~breiman/RandomForests/">stat.berkeley.edu/~breiman/RandomForests</a>; la descripción del propio Breiman del algoritmo, la estimación OOB y las medidas de importancia.<sup class="c"><a href="#r35">[35]</a></sup></li>
<li>Visualizaciones interactivas de árboles binarios de búsqueda, AVL y rojo–negro (University of San Francisco, D. Galles) — <a href="https://www.cs.usfca.edu/~galles/visualization/Algorithms.html">cs.usfca.edu/~galles/visualization</a>; útiles para ver cómo ocurren las rotaciones.</li>
<li>UCI Machine Learning Repository — <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a>; fuente original de los datos de cáncer de mama y de viviendas.</li>
</ul>
